# Insurance Policy RAG System — v2.6 (LangGraph)

**v2.6 = v2.3's answer pipeline + a deterministic evaluation workflow + a BM25 switch.** With the
default settings the answers are the same as v2.3. RAGAS and Langfuse (added in v2.4) are set aside
until this workflow is reliable; they remain available in v2.4/v2.5 for later.

| Change | Why | Section |
|---|---|---|
| **BM25 switch** (`use_bm25`, default `True`); retrieval sizes are read from `CONFIG` when a question is asked | Lets measurement decide whether BM25 earns its place. Experiments: dense only (top 32 and top 64) and BM25 only | 0.1, 5 |
| **Test set of 94 questions** across 10 question types, every key fact checked against the policy | The 10-question gold set is too small, and 3 of its 9 answerable questions expect keywords that are not in their own gold sections | 10.3 |
| **Evaluation workflow**: retrieval, answer correctness, faithfulness (figures and section numbers the context does not contain), reliability; saved runs that reload in a later session; paired comparisons with 95% intervals | Every change is measured against the same baseline on the same questions, with no API keys | 10.4–10.5 |
| **Errors and out-of-memory retries are recorded in the pipeline state** (`error`, `oom_retry`) | A failing node looked like a refusal, and an answer from the shortened retry context looked like a normal one | 8 |
| **Three measured fixes, off by default**: precise title pinning (`title_pinning_rules = "v2.5"`), stemmed BM25 (`bm25_stemming = True`) and continuation sections (`continuation_sections = True`) | Title pinning brought unrelated sections into 11 of 94 questions ("claims" pinned all of Section 11); BM25 missed "notified" vs "Notification"; the Cumulative Bonus notes (3.9, 3.10) are numbered as separate sections, so neither pinning nor expansion brings them with 3.8 | 6.1, 4.1 |

**What the recorded runs say about poor answers:** in 8 of 9 recorded answers, the share of key
facts in the answer equals the share in its context, and the one invented clause appeared where the
context held 17% of the facts. The main cause is evidence that never reaches the model; generation
errors are rare. Details: `docs/improvement_plan.md`.

**How to use it:** *Run all* runs 13 retrieval experiments in Section 10.5 (about 15–20 minutes on a
T4). Then set `EVAL_PLAN["full"] = ["baseline"]` for the answer baseline (30–45 minutes) and save
everything with `export_runs()`. Make a change the default only when it is better with no clear
regression.

---

## v2.3 notes

**v2.3 = v2.2 + answer-quality fixes.** Each one was traced from the recorded v2.2 answers
(`examples/sample_outputs.md`), whose sources and reranker scores show what reached the model.

| Weak answer in v2.2 | Measured cause | v2.3 fix |
|---|---|---|
| 24-month waiting period: rule given, conditions never listed | The retrieved 10.1 chunk ends "List of specific diseases/procedures is provided below"; the list (10.1.1) has none of the question's words, so search never returns it | **Neighbour expansion**: a selected chunk brings the chunk it continues into (same section or a sub-section), and follows a "…provided below" pointer one step further |
| Plus Benefit: misses the core rule (50% added per renewal, up to 100%) | Section 4.4 sits right before the retrieved 4.4.1 Notes but was outranked by Global Health Cover chunks | **Title pinning**: when a section heading ("Plus Benefit") appears in the question, that section and its sub-sections go into the context first; expansion also adds a sub-section's parent heading chunk |
| Plus Benefit ends with an unrelated tax paragraph | A Section 9 chunk scoring −5.77 was used to fill the 1,500-token budget | **Relevance floor**: reranked chunks below −5.0 (the gate threshold) are no longer used as filler |
| Plan comparison mixes Year 1 and Year 3 figures | The Year 2 table (2.2) was missing | Expansion adds the neighbouring plan table, plus a prompt rule: one year's table per comparison |
| Standard exclusions stop at Excl02 | Top chunk was the product overview; 10.1 ranked third | Title pinning ("Standard Exclusions" → 10.1 and sub-sections). The full list is ~2,060 tokens — more than the 1,500-token budget — so the prompt asks the model to say when a list continues beyond the context |
| Pre/post-hospitalization list Section 3.1's items as inclusions; a Global Health Cover exclusion (4.9.2) applied to base cover | Right sections retrieved; the model borrowed across sections | **Scope rules** in the prompt (switchable, A/B-testable in Section 10.2) |

Also: section titles now stop where the body text begins ("3.5 Pre-Hospitalization Expenses",
not "Pre-Hospitalization Expenses The Company shall indemnify…"), which title pinning needs and
which makes the labels cleaner; consecutive chunks are merged into one passage per group with the
repeated overlap sentence removed.

Everything is switchable in `CONFIG` (`title_pinning`, `neighbour_expansion`, `seed_score_floor`,
`prompt_version`), so each fix can be measured on its own with Sections 10 and 10.1.

---

## v2.2 notes

**v2.2 = v2.1 + fixes for CUDA out-of-memory on some questions (Colab T4).**

| Cause | Fix |
|---|---|
| v2.1 filled the context *past* 1,500 tokens (the chunk that crossed the budget was kept) and the `[Section …]` labels were not counted, so prompts reached ~2,180 tokens vs ~1,600 in the original notebook — on a T4 already ~95% full with Mistral fp16 + reranker + embedder | **Hard 1,500-token cap that includes the labels** → prompts are back to the original's size (best-case evidence coverage stays 98–100%) |
| Freed GPU memory stays fragmented between stages | `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` + `torch.cuda.empty_cache()` after reranking and after each answer |
| One OOM failed the whole answer | On OOM: free memory, retry once with the top-ranked chunks (≤1,000 tokens), print a warning |
| No visibility into GPU headroom | Memory report after loading; warning if any LLM layers were pushed to CPU |
| Still OOM after all of the above | `CONFIG["llm_load_in_4bit"] = True` (off by default — it changes the model's numbers slightly) |

---

## v2.1 notes

**v2.1 = v2 + targeted fixes for the answer-quality regression versus the original notebook.**
Each fix below was traced by rebuilding the policy text from the original notebook's printed
chunks (all 84 reproduced exactly), running v2's own parsing/chunking code on it, and measuring
how much of the evidence the original answers actually used v2 could still deliver.

| # | Cause (measured) | Effect | v2.1 fix |
|---|---|---|---|
| 1 | Neighbor dedup drops adjacent subchunks | 9 of 10 original contexts used adjacent subchunks; adjacent v2 chunks share only ~29% (one sentence), so dedup threw away mostly new text. Best-case evidence coverage fell to 64% / 67% / 84% / 92% on 4 questions | `dedup_neighbors=False` |
| 2 | Same chunk *count* (5) with smaller chunks (avg 181 vs 288 tok) | Context shrank from ~1,500 to ~900 tokens; pre/post-hospitalization capped at 84% even without dedup | Fill context to the original's **1,500-token budget** (≈8 chunks) |
| 3 | Candidate pool cut to fused top-20 | Reranker chose from ~3,600 tokens of candidates vs ~5,800 originally; when legs disagree only ~10 dense candidates survive | Rerank the full union of dense top-32 + BM25 top-32 |
| 4 | Per-chunk score floor (−5) | Not in the original; your v1 log shows it dropping chunks (Plus Benefit, Excl06) | Floor off; gate on best score unchanged |
| 5 | Section titles dropped their first word | Labels read "Benefit On Renewal…" for **Plus** Benefit, "Travel Secure" for **Overseas** Travel Secure, "Expenses…" for **Pre/Post-Hospitalization** — in the embedded text, BM25 text and the LLM's context labels | Title starts at the heading's first word, ends at the heading sentence |
| 6 | Double BOS (`<s><s>[INST]`) | Rendered chat template already contains `<s>`; v2 tokenized it with special tokens again | `add_special_tokens=False` + runtime assertion |
| 7 | "Refer Annexure C" parsed as a heading | Blocked real Annexure B detection; 17-token junk section | `refer` added to reference words |
| 8 | Tail-merge repeated the overlap sentence | Duplicate sentence inside merged chunks | Merge only the new sentences |

**Generation wording:** the prompt instructions are restored to the original notebook's exact text
(v2 had added "ignore unrelated sections" and "cite … for each fact"). The section-labeled context
blocks and clean answer extraction (v2's formatting improvements) are kept. Whether the chat
template or the raw prompt gives better answers could not be measured offline — Section 10.2 runs
that A/B on identical contexts.

**New:** Section 10.1 checks every answer's context against the evidence the *original* notebook's
answers used (embedded fixture), so the regression is measured, not eyeballed.

Pipeline: **ingest → parse sections → sentence-pack chunks → index (dense + BM25) → hybrid retrieve (union) → rerank → budgeted context → gate → answer / no-answer**


## How to run this evaluation (read this first)

This is **notebook v2.6, unchanged**, plus **Section 12**, which collects every result into one zip
file for review. Nothing in Section 12 changes how questions are answered.

1. Open the notebook in Google Colab and choose a GPU: *Runtime → Change runtime type → T4 GPU*.
2. Optional: adjust `EVAL_RUN` in Section 12.1. The defaults collect everything needed.
   For a protected long run, set `"save_to_drive": True` (Colab asks once for Drive access).
3. *Runtime → Run all*. When Section 1 asks, upload the policy `.txt` file.
4. Keep the tab open. It takes about **1.5–2 hours** on a T4: model downloads ~10 min, Sections 9–10.1
   ~15 min, 13 retrieval experiments ~20 min, full answers to 94 questions ~45–55 min.
5. At the end, `rag_eval_results_<date>.zip` downloads automatically. **Send that zip back.**

**If something stops:**
- *The run stops but the runtime is still alive:* re-run Section 12.3; it resumes from the last
  checkpoint (every 10 questions), then run 12.4.
- *The runtime disconnects:* with `save_to_drive` on, *Run all* again; the full run resumes from the
  checkpoint in Drive. Without it, the full run starts over.
- *CUDA out of memory:* *Runtime → Restart session*, then *Run all*. If it repeats, check the `[GPU …]`
  line printed after model loading and tell me what it says. Do not switch to 4-bit for this run: it
  would change the answers being measured.
- Any other error: send the error text and, if the zip was produced, the zip.

## 0. Setup and configuration

In [ ]:
# ================================================================
# 0) Install dependencies
# ================================================================
!pip install -q tiktoken sentence-transformers transformers faiss-cpu accelerate rank_bm25 langgraph pandas
!pip install -q snowballstemmer==3.0.1   # v2.5: only used when CONFIG['bm25_stemming'] = True

import os
# v2.2: reduce CUDA memory fragmentation (the fix PyTorch's own OOM message suggests).
# Must be set before torch first touches the GPU: if torch was already imported in this
# runtime, use Runtime -> Restart session, then run from the top.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re, json, torch, faiss, traceback
import numpy as np
import pandas as pd
from pathlib import Path
from typing import TypedDict, List

IN_COLAB = False
try:
    from google.colab import files  # noqa
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | CUDA: {torch.cuda.is_available()}")


In [ ]:
# ================================================================
# 0.1) Configuration — every tunable in one place
# ================================================================
CONFIG = {
    # Chunking
    "chunk_size_tokens": 300,
    "table_chunk_tokens": 600,     # larger budget for table-like sections
    "min_tail_tokens": 50,         # tails below this are MERGED into previous chunk
    "title_max_words": 12,         # max words in a section title (v2.3 titles stop at the body text)

    # Models
    "embed_model": "BAAI/bge-small-en-v1.5",
    "bge_query_prefix": "Represent this sentence for searching relevant passages: ",
    "use_query_prefix": True,      # A/B toggle
    "use_section_prefix": True,    # prepend the section label to embedded + BM25 text (A/B toggle)
    "reranker_model": "BAAI/bge-reranker-base",
    "llm_model": "mistralai/Mistral-7B-Instruct-v0.3",

    # Retrieval (v2.1): candidate pool sized to the ORIGINAL's candidate text volume.
    # Original: 20 candidates x ~290 tokens ~= 5,800 tokens. v2 chunks average ~180 tokens -> 32.
    # Every candidate in the union of both legs is reranked (v2 truncated the fused list to 20).
    "k_dense": 32,
    "k_bm25": 32,
    "rrf_k": 60,                   # now only orders the union before reranking
    "use_bm25": True,              # v2.6: False = dense search only (the BM25 index is still built)
    "bm25_stemming": False,        # v2.5 option: stem words for BM25 ("notified" ~ "notification")

    # Context
    "context_token_budget": 1500,  # HARD cap incl. section labels (v2.2) = original prompt size
    "title_pinning": True,         # v2.3: sections whose heading appears in the question go first
    "neighbour_expansion": True,   # v2.3: add the chunk a selected chunk continues into / depends on
    "seed_score_floor": -5.0,      # v2.3: reranked chunks below this are not used to fill the context
    "title_pinning_rules": "v2.3", # v2.5 option: "v2.5" = precise title pinning (see Section 6.1)
    "continuation_sections": False, # v2.6 option: notes numbered as separate sections join their section

    # Gate
    "rerank_score_threshold": -5.0,  # refuse only if the BEST reranker score is below this

    # Generation
    "max_new_tokens": 1000,
    "use_chat_template": True,     # A/B toggle: False = original notebook's raw prompt format
    "prompt_version": "v2.3",      # "v2.3" = original instructions + scope rules; "original" = v2.2

    # Memory (v2.2)
    "oom_retry_context_tokens": 1000,  # on CUDA OOM, retry once with the top chunks up to this size
    "llm_load_in_4bit": False,         # last resort if OOM persists: ~4 GB instead of ~14.5 GB

    # Persistence
    "index_path": "policy_index.faiss",
    "chunks_path": "chunks.json",
}

NO_ANSWER_TEXT = "Not specified in the provided policy text."

def is_no_answer(ans: str) -> bool:
    """The original prompt wraps the refusal in curly quotes; accept either form."""
    return ans.strip().strip('"“”').strip() == NO_ANSWER_TEXT


## 1. Ingest and preprocess

In [ ]:
# ================================================================
# 1) Upload (Colab) or set a path, normalize, strip page headers
# ================================================================
SOURCE_PATH = None  # set a path string here to skip the upload dialog

if SOURCE_PATH is None and IN_COLAB:
    print("Upload your policy .txt file")
    uploaded = files.upload()
    SOURCE_PATH = list(uploaded.keys())[0]

assert SOURCE_PATH, "Set SOURCE_PATH to your policy .txt file"

text = Path(SOURCE_PATH).read_text(encoding="utf8", errors="ignore")
text = re.sub(r"\r\n?", "\n", text)

# NEW: strip repeated page headers/footers from the PDF->txt conversion.
# These appear mid-sentence in the body and previously spawned a phantom
# 'my' section. Add any other repeated header lines you find to this list.
PAGE_HEADER_PATTERNS = [
    r"HDFC ERGO General Insurance Company Limited",
    r"my:\s*Optima Secure\s*-\s*Prospectus",
]
for pat in PAGE_HEADER_PATTERNS:
    text = re.sub(pat, " ", text)

text = re.sub(r"\n{3,}", "\n\n", text)
text = re.sub(r"[ \t]{2,}", " ", text)
text = text.strip()
print(f"File: {SOURCE_PATH} | length after cleaning: {len(text):,} characters")


## 2. Section parsing (sequence-aware)

The v1 regex only matched headings at line starts; in this document most headings occur
**mid-line inside flowing text**, so 14+ real sections were lumped under one label. This parser
finds candidates anywhere, then filters with three heuristics:

1. **Reference rejection** — "under Section 3.1", "as set out in Section 4" are cross-references,
   not headings (preceding word check).
2. **Title-case check** — real headings are followed by a Title-Case or numeric token.
3. **Monotone ladder** — genuine section numbers only ever advance (1 → 1.1 → 2 → 2.1 …);
   any candidate that goes backwards is a reference.

Annexures (letter IDs) are tracked on their own advancing ladder.

In [ ]:
# ================================================================
# 2) Detect section headings
# ================================================================
sec_pat = re.compile(r"(Section)\s+(\d+(?:\.\d+)*)\.?\s+(\S+)")
anx_pat = re.compile(r"(Annexure|Appendix)\s+([A-Z])\b\s*[-.]?\s*(\S*)")
REF_WORDS = ("under", "per", "in", "to", "against", "with",
             "below", "above", "of", "and", "see", "refer")   # v2.1: + "refer" ("Refer Annexure C")

def num_tuple(s: str):
    return tuple(int(x) for x in s.split("."))

candidates = sorted(
    [(m.start(), "sec", m) for m in sec_pat.finditer(text)] +
    [(m.start(), "anx", m) for m in anx_pat.finditer(text)]
)

headings, last_num, last_anx = [], (), ""
for pos, kind, m in candidates:
    nxt = m.group(3)
    if nxt and not (nxt[0].isupper() or nxt[0].isdigit()):
        continue  # followed by lowercase -> in-sentence reference
    before = text[max(0, pos - 25):pos].lower().split()
    if before and before[-1] in REF_WORDS:
        continue  # "under Section 3.1" etc.
    if kind == "sec":
        n = num_tuple(m.group(2))
        if last_num and n <= last_num:
            continue  # goes backwards -> reference
        last_num = n
        # v2.1: the title starts AT the heading's first word (group 3); v2 started after it,
        # dropping "Plus", "Overseas", "Pre-Hospitalization", "Procedure", ... from every label
        headings.append({"start": pos, "section_id": m.group(2), "title_start": m.start(3)})
    else:
        # v2.1: "Section 15.1. Annexure A - ..." -> the Annexure name is that section's title
        if headings and pos - headings[-1]["start"] < 20:
            continue
        if last_anx and m.group(2) <= last_anx:
            continue
        last_anx = m.group(2)
        headings.append({"start": pos, "section_id": f"Annexure-{m.group(2)}", "title_start": m.end(2)})

print(f"Detected {len(headings)} sections")


In [ ]:
# ================================================================
# 2.1) VERIFY the section ladder before proceeding
#      (everything downstream depends on this being right)
# ================================================================
ladder = [h["section_id"] for h in headings]
print("Section ladder:")
print(" -> ".join(ladder))

# Expect a clean monotone sequence like:
# 1 -> 1.1 -> 2 -> 2.1 -> 2.2 -> ... -> 15.1 -> Annexure-A -> Annexure-B
# If sections you know exist are MISSING, check whether a page-header pattern
# splits the heading text, or add the offending preceding word to REF_WORDS.


In [ ]:
# ================================================================
# 2.2) Build sections with clean short titles
# ================================================================
TITLE_WORDS = CONFIG["title_max_words"]

# v2.3: titles stop where the body text begins ("Plus Benefit", not "Plus Benefit On Renewal of...")
BODY_START = {"The", "This", "These", "On", "If", "An", "Any", "All", "Our", "You", "We",
              "It", "For", "After", "Section", "Please", "Pre-Policy", "Note:", "my:"}
BODY_VERBS = {"is", "are", "shall", "will", "may"}
LIST_MARK = re.compile(r"^(?:[a-z]|[ivx]{1,4}|[A-Z])[.)]$|^[ivx]{1,4}\)$|^·$|^\d+\.$")

def make_title(s: str, max_words: int = TITLE_WORDS, stop_on_verbs: bool = True) -> str:
    """Heading words, stopping where the body text begins (best effort; labels only)."""
    words = s.split()[:max_words + 2]
    out = []
    for i, w in enumerate(words[:max_words]):
        nxt = words[i + 1] if i + 1 < len(words) else ""
        nxt2 = words[i + 2] if i + 2 < len(words) else ""
        if out and (w in BODY_START or LIST_MARK.match(w) or (stop_on_verbs and w in BODY_VERBS)):
            break
        if out and w == "In" and nxt[:1].islower():          # "In case ..." / "In the event ..."
            break
        if out and w[:1].isupper() and nxt in ("and", "with") and nxt2[:1].islower():
            break                                             # "... Notice | Policy and any ..."
        out.append(w)
        if w.endswith(".") and not re.fullmatch(r"\(?[A-Za-z]{1,3}\.\)?", w):
            break
        if len(out) >= 2 and (w.endswith(":") or w.endswith(",")):
            break
        if len(out) >= 2 and w.endswith(")") and "(" in " ".join(out):
            break                                             # "... Cover (Emergency Treatments Only)"
    return " ".join(out).strip(" .:-,")


sections = []
for i, h in enumerate(headings):
    start = h["start"]
    end = headings[i + 1]["start"] if i + 1 < len(headings) else len(text)
    body = text[start:end].strip()
    if h["section_id"].startswith("Annexure"):    # annexure names are longer and contain verbs
        title = make_title(text[h["title_start"]:end], max_words=16, stop_on_verbs=False)
    else:
        title = make_title(text[h["title_start"]:end])
    sections.append({
        "section_id": h["section_id"],
        "section_title": title,
        "text": body,
    })

# Preamble before the first heading (product intro) becomes its own section
# (any non-trivial preamble is kept — dropping it would lose text)
if headings and headings[0]["start"] > 50:
    sections.insert(0, {
        "section_id": "0",
        "section_title": "Introduction",
        "text": text[:headings[0]["start"]].strip(),
    })
if not headings:
    print("WARNING: no sections detected; using whole document as one section.")
    sections = [{"section_id": "0", "section_title": "Document", "text": text}]

print(f"{len(sections)} sections ready")


## 3. Sentence-aware chunking

Raw token windows cut mid-number ("₹2,00,00,000" → ",00,000. All Insured…"). This packer splits
each section into sentences and packs them greedily to the token budget with **one-sentence
overlap**. Undersized tails are **merged into the previous chunk** rather than dropped (dropping
lost text). Table-like sections (the plan-comparison illustrations) get a larger budget so
comparison rows stay together, at the cost of reranker/embedder truncation beyond 512 tokens —
BM25 still covers their full text.

Each chunk stores two texts: `text` (raw, shown to the LLM) and `embed_text` (prefixed with
`Section {id} – {title}:`), so mid-section chunks embed with their topic and section numbers are
BM25-searchable from every chunk.

In [ ]:
# ================================================================
# 3) Chunking
# ================================================================
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

def tok_len(s: str) -> int:
    return len(enc.encode(s))

def split_sentences(t: str):
    # Split after . or ; when followed by uppercase / digit / bullet / rupee / paren
    parts = re.split(r"(?<=[.;])\s+(?=[A-Z0-9\u00b7\u20b9(])", t)
    return [p.strip() for p in parts if p.strip()]

def looks_like_table(sec_text: str) -> bool:
    # Plan-comparison illustrations: dense with rupee amounts and 'for <Plan>'
    rupee_hits = len(re.findall(r"\u20b9|Rs\.?\s*\d", sec_text))
    for_hits = len(re.findall(r"\bfor (Suraksha|Secure|Super|Global)", sec_text))
    return rupee_hits >= 8 and for_hits >= 4

def pack_section(sec: dict, max_tok: int, min_tail: int):
    sents = split_sentences(sec["text"])
    packed, cur, cur_tok, fresh = [], [], 0, 0   # fresh = sentences not yet in packed[-1]
    for s in sents:
        st = tok_len(s)
        if fresh and cur_tok + st > max_tok:
            packed.append(" ".join(cur))
            cur = [cur[-1]]              # one-sentence overlap
            cur_tok = tok_len(cur[0])
            fresh = 0
        cur.append(s)
        cur_tok += st
        fresh += 1
    if fresh:
        new = cur[len(cur) - fresh:]
        if packed and tok_len(" ".join(new)) < min_tail:
            # v2.1: merge only the NEW sentences; the carried overlap sentence already
            # ends packed[-1] (v2 appended it a second time)
            packed[-1] = packed[-1] + " " + " ".join(new)
        else:
            packed.append(" ".join(cur))
    return packed

final_chunks = []
for sec in sections:
    is_table = looks_like_table(sec["text"])
    budget = CONFIG["table_chunk_tokens"] if is_table else CONFIG["chunk_size_tokens"]
    pieces = pack_section(sec, budget, CONFIG["min_tail_tokens"])
    for j, piece in enumerate(pieces, start=1):
        label = f"Section {sec['section_id']} - {sec['section_title']}"
        final_chunks.append({
            "idx": len(final_chunks),             # v2.3: position in document order
            "is_table": is_table,                 # v2.3: plan-comparison table section
            "section_id": sec["section_id"],
            "section_title": sec["section_title"],
            "subchunk_id": j,
            "token_count": tok_len(piece),
            "text": piece,                        # raw text -> LLM context
            "embed_text": f"{label}: {piece}",    # labeled text -> embeddings + BM25
        })

print(f"Chunking complete -> {len(final_chunks)} chunks from {len(sections)} sections")


In [ ]:
# ================================================================
# 3.1) Inspect chunks (bounded preview)
# ================================================================
num_chunks = len(final_chunks)
avg_tokens = sum(c["token_count"] for c in final_chunks) / max(num_chunks, 1)
n_sections = len({c["section_id"] for c in final_chunks})
print(f"Total chunks: {num_chunks} | avg tokens: {avg_tokens:.1f} | distinct sections: {n_sections}")

for i, c in enumerate(final_chunks[:5]):
    print(f"\n=== Chunk {i+1} | Sec {c['section_id']} - {c['section_title']} "
          f"| sub {c['subchunk_id']} | {c['token_count']} tok ===")
    print(c["text"][:400], "...")

df_chunks = pd.DataFrame([{
    "Section_ID": c["section_id"],
    "Title": c["section_title"][:40],
    "Sub": c["subchunk_id"],
    "Tokens": c["token_count"],
    "Preview": c["text"][:100].replace("\n", " "),
} for c in final_chunks])
df_chunks.head(10)


## 4. Indexing: dense (FAISS) + lexical (BM25)

Both indexes are built over `embed_text` (section-labeled), so section numbers and titles are
searchable in every chunk. `IndexFlatIP` (exact) is deliberate at this corpus size. Both
artifacts are persisted, separating ingestion from serving.

In [ ]:
# ================================================================
# 4) Dense embeddings + FAISS
# ================================================================
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer(CONFIG["embed_model"])

embed_texts = [c["embed_text"] if CONFIG["use_section_prefix"] else c["text"]
               for c in final_chunks]
embeddings = embed_model.encode(
    embed_texts, batch_size=32, convert_to_numpy=True, show_progress_bar=True
).astype("float32")

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("FAISS index built:", index.ntotal)


In [ ]:
# ================================================================
# 4.1) BM25 lexical index
# ================================================================
from rank_bm25 import BM25Okapi

_stemmer = None

def _stem(w: str) -> str:
    global _stemmer
    if _stemmer is None:
        import snowballstemmer
        _stemmer = snowballstemmer.stemmer("english")
    w = re.sub(r"ised$", "ized", re.sub(r"isation", "ization", w))   # British -> American spelling
    return _stemmer.stemWord(w)

def bm25_tokenize(s: str):
    toks = re.findall(r"[a-z0-9.]+", s.lower())  # keeps '4.7.1' and 'excl06' intact
    if CONFIG["bm25_stemming"]:                  # v2.5 option; numbers and codes are not stemmed
        toks = [t if re.search(r"\d", t) else _stem(t) for t in toks]
    return toks

def build_bm25():
    """(Re)build the BM25 index with the current tokenizer settings."""
    global bm25
    bm25 = BM25Okapi([bm25_tokenize(t) for t in embed_texts])

build_bm25()
print("BM25 index built over", len(embed_texts), "chunks")


In [ ]:
# ================================================================
# 4.2) Persist index + chunks
# ================================================================
faiss.write_index(index, CONFIG["index_path"])
Path(CONFIG["chunks_path"]).write_text(json.dumps(final_chunks, ensure_ascii=False))
print(f"Saved {CONFIG['index_path']} and {CONFIG['chunks_path']}")

# To reload without re-ingesting:
# index = faiss.read_index(CONFIG["index_path"])
# final_chunks = json.loads(Path(CONFIG["chunks_path"]).read_text())
# embed_texts = [c["embed_text"] if CONFIG["use_section_prefix"] else c["text"] for c in final_chunks]
# bm25 = BM25Okapi([bm25_tokenize(t) for t in embed_texts])


## 5. Hybrid retrieval (dense + BM25) — full union is reranked

v2.1: v2 truncated the RRF-fused list to 20, so when the two legs disagreed only ~10 dense
candidates reached the reranker (the original reranked 20). Now both legs return 32 candidates
(sized so the candidate *text volume* matches the original's 20 × ~290 tokens) and the whole
union is reranked. RRF only orders the union.

In [ ]:
# ================================================================
# 5) Hybrid retrieval
# ================================================================
def hybrid_retrieve(query: str, k_dense: int = None, k_bm25: int = None):
    # v2.6: sizes are read from CONFIG when called (they used to be fixed when the cell ran), and
    # CONFIG["use_bm25"] = False turns the lexical leg off. Defaults give the same candidates as before.
    k_dense = CONFIG["k_dense"] if k_dense is None else k_dense
    k_bm25 = (CONFIG["k_bm25"] if k_bm25 is None else k_bm25) if CONFIG["use_bm25"] else 0
    assert k_dense > 0 or k_bm25 > 0, "both retrieval legs are switched off"

    # Dense leg — BGE query instruction prefix (passages are embedded without it)
    dense_ranked = []
    if k_dense > 0:
        q_text = (CONFIG["bge_query_prefix"] + query) if CONFIG["use_query_prefix"] else query
        q_emb = embed_model.encode([q_text], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(q_emb)
        _, I = index.search(q_emb, k_dense)
        dense_ranked = [int(i) for i in I[0] if i != -1]

    # Lexical leg
    bm25_ranked = []
    if k_bm25 > 0:
        scores = bm25.get_scores(bm25_tokenize(query))
        bm25_ranked = [int(i) for i in np.argsort(scores)[::-1][:k_bm25]]

    # Reciprocal rank fusion — orders the union; nothing is cut (v2.1)
    K = CONFIG["rrf_k"]
    rrf = {}
    for rank, idx in enumerate(dense_ranked):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (K + rank)
    for rank, idx in enumerate(bm25_ranked):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (K + rank)

    fused = sorted(rrf, key=rrf.get, reverse=True)
    return [final_chunks[i] for i in fused]


## 6. Reranker

The cross-encoder scores every candidate against the question. Context selection moved to
Section 6.1 in v2.3.

In [ ]:
# ================================================================
# 6) Reranker
# ================================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
tok_rerank = AutoTokenizer.from_pretrained(CONFIG["reranker_model"])
reranker = AutoModelForSequenceClassification.from_pretrained(CONFIG["reranker_model"])
reranker.to(device).eval()

def rerank_all(query: str, docs: List[str], batch_size: int = 16):
    """Score every doc against the query (batched: the union can hold up to ~64 candidates)."""
    out = []
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        inputs = tok_rerank([query] * len(batch), batch,
                            padding=True, truncation=True, max_length=512,
                            return_tensors="pt").to(device)
        with torch.no_grad():
            out.append(reranker(**inputs).logits.squeeze(-1).float().cpu().numpy().reshape(-1))
    scores = np.concatenate(out) if out else np.array([])
    order = scores.argsort()[::-1]
    return order, scores

def free_gpu():
    """Return cached blocks to the allocator between pipeline stages (v2.2)."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def block_label(c: dict) -> str:
    return f"[Section {c['section_id']} - {c['section_title']}]"

def block_tokens(c: dict) -> int:
    """Tokens this chunk adds to the prompt: label + newline + text + separator."""
    if "block_tokens" not in c:
        c["block_tokens"] = tok_len(block_label(c)) + c["token_count"] + 2
    return c["block_tokens"]



## 6.1 Context building (v2.3; v2.5 adds precise title pinning as an option)

The context is assembled in three steps, under the same hard 1,500-token cap as v2.2:

1. **Title pinning** — if a section heading appears in the question ("Plus Benefit",
   "Standard Exclusions", "Pre-Hospitalization Expenses"), that section and its sub-sections go
   first. A heading contained in a longer matching heading is ignored, so "Hospitalization
   Expenses" (3.1) does not get pinned for a pre-hospitalization question.
2. **Reranked seeds** — chunks the reranker scored at least `seed_score_floor` (−5.0), best first.
   Lower-scoring chunks are no longer used as filler.
3. **Neighbour expansion** — each pinned section or seed brings the text it depends on: the next
   chunk of the same section or a sub-section (one more if that chunk ends with "…provided below"),
   the previous chunk when it is the same section or the parent heading, and the neighbouring plan
   table.

Each group is shown to the model as one passage, labelled with every section it covers, with the
sentence repeated between consecutive chunks removed.

In [ ]:
# ================================================================
# 6.1) Context building — title pinning, relevance floor, neighbour expansion
# ================================================================
_TITLE_STOP = {"of", "for", "the", "a", "an", "in", "to", "and", "on", "with", "under", "by",
               "procedure", "list", "section", "notes", "note", "rs", "my"}
_POINTER = re.compile(r"(below|following|as follows|hereunder)\W*$|:\s*$", re.I)

def _words(s: str):
    return re.findall(r"[a-z0-9]+", s.lower())

def title_core(title: str) -> frozenset:
    title = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", title)      # "(CB)", "(Rs.)", "[Applicable ...]"
    return frozenset(w for w in _words(title) if w not in _TITLE_STOP)

SECTION_CORES = {s["section_id"]: title_core(s["section_title"]) for s in sections}

def title_matches(query: str, max_sections: int = 2):
    """Sections whose heading words all appear in the question ("Plus Benefit",
    "Standard Exclusions", "Pre-Hospitalization Expenses"). A match contained in a
    longer match is dropped, so "Hospitalization Expenses" (3.1) loses to
    "Pre-Hospitalization Expenses" (3.5)."""
    q = set(_words(query))
    hits = [sid for sid, core in SECTION_CORES.items() if core and core <= q]
    hits = [a for a in hits if not any(SECTION_CORES[a] < SECTION_CORES[b] for b in hits)]
    hits.sort(key=lambda sid: -len(SECTION_CORES[sid]))
    return hits[:max_sections]

def _parent(sid: str):
    return sid.rsplit(".", 1)[0] if "." in sid else None

def _is_child(a: str, b: str) -> bool:          # a is inside b (10.1.1 inside 10.1)
    return a.startswith(b + ".")

def _same_table_family(a: dict, b: dict) -> bool:  # plan tables 2.1 / 2.2 / 2.3 (Year 1-3)
    return a.get("is_table") and b.get("is_table") and _parent(a["section_id"]) == _parent(b["section_id"])

def _group_for(i: int):
    """A seed chunk plus the text it depends on: the rest of its sentence run (next chunk of
    the same section or a sub-section, following a "...provided below" pointer once more),
    its own section heading (previous chunk of the same section or its parent section),
    and the neighbouring year of a plan table."""
    seed, out = final_chunks[i], [i]          # priority order: seed first
    if i + 1 < len(final_chunks):
        nxt = final_chunks[i + 1]
        if (nxt["section_id"] == seed["section_id"] or _is_child(nxt["section_id"], seed["section_id"])
                or _same_table_family(seed, nxt) or _continues(nxt["section_id"], seed["section_id"])):
            out.append(i + 1)
            if _POINTER.search(nxt["text"].strip()) and i + 2 < len(final_chunks):
                nn = final_chunks[i + 2]
                if nn["section_id"] == nxt["section_id"] or _is_child(nn["section_id"], nxt["section_id"]) \
                        or _is_child(nn["section_id"], seed["section_id"]):
                    out.append(i + 2)
    if i > 0:
        prv = final_chunks[i - 1]
        if (prv["section_id"] == seed["section_id"] or prv["section_id"] == _parent(seed["section_id"])
                or _same_table_family(seed, prv) or _continues(seed["section_id"], prv["section_id"])):
            out.append(i - 1)
    return out

def _pinned_group(sid: str):
    """All chunks of a title-matched section, followed by its sub-sections, in document order."""
    idxs = [c["idx"] for c in final_chunks if c["section_id"] == sid]
    j = idxs[-1] + 1
    while j < len(final_chunks) and _is_child(final_chunks[j]["section_id"], sid):
        idxs.append(j)
        j += 1
    if CONFIG.get("continuation_sections"):     # v2.6 option: notes numbered as separate sections
        while j < len(final_chunks) and CONTINUATION_OF.get(final_chunks[j]["section_id"]) == sid:
            idxs.append(j)
            j += 1
    return idxs

def build_context(query: str, candidates: List[dict], scores):
    """Returns (groups, score_of) — groups are lists of chunk indices in document order,
    most relevant group first. Fill order: title-matched sections, then reranked chunks
    scoring at least the relevance floor, each with its dependent neighbours. Hard token cap
    (section labels included)."""
    budget, floor = CONFIG["context_token_budget"], CONFIG["seed_score_floor"]
    score_of = {c["idx"]: float(s) for c, s in zip(candidates, scores)}
    ranked = sorted(score_of, key=score_of.get, reverse=True)
    plan = [_pinned_group(sid) for sid in title_matches_active(query)] if CONFIG["title_pinning"] else []
    plan += [_group_for(i) if CONFIG["neighbour_expansion"] else [i]
             for i in ranked if score_of[i] >= floor]
    used_tok, taken, groups = 0, set(), []
    for g in plan:
        group = []
        for i in g:
            if i in taken:
                continue
            cost = block_tokens(final_chunks[i])
            if used_tok + cost > budget:
                if group or groups:       # budget full: stop growing this group
                    break
                continue
            group.append(i); taken.add(i); used_tok += cost
        if group:
            groups.append(sorted(group))
        if used_tok >= budget - 40:
            break
    return groups, score_of

def merge_group_text(group):
    """Join consecutive chunks of one group into a single passage, dropping the sentence a
    chunk repeats from the previous chunk of the same section (one-sentence overlap)."""
    parts = []
    for k, i in enumerate(group):
        t = final_chunks[i]["text"]
        if k and final_chunks[group[k - 1]]["section_id"] == final_chunks[i]["section_id"] \
                and group[k - 1] == i - 1:
            sents = split_sentences(t)
            prev_last = split_sentences(final_chunks[i - 1]["text"])[-1]
            if sents and sents[0] == prev_last:
                t = " ".join(sents[1:])
        parts.append(t)
    return " ".join(p for p in parts if p)

def group_label(group):
    seen, labels = set(), []
    for i in group:
        c = final_chunks[i]
        if c["section_id"] not in seen:
            seen.add(c["section_id"])
            title = c["section_title"]
            if CONFIG.get("continuation_sections") and c["section_id"] in CONTINUATION_OF:
                title = f"{_TITLE_OF[CONTINUATION_OF[c['section_id']]]} (continued)"   # v2.6 option
            labels.append(f"Section {c['section_id']} - {title}")
    return "[" + " → ".join(labels) + "]"

# v2.5 title-pinning rules (CONFIG["title_pinning_rules"] = "v2.5"); "v2.3" keeps the old matcher.
# Measured on the 92-question test set, the v2.3 matcher's pinned groups brought unrelated
# sections into 9 questions (8,574 tokens in total, before the 1,500-token cap); v2.5: 4 questions,
# 1,090 tokens, all related sections, and no question lost its gold section. Causes:
#  - one-word headings pin on common words: "claims" pins all of Section 11 (1,774 tokens with its
#    sub-sections), "exclusions" would pin all of Section 10;
#  - the Global Health Cover headings differ only inside parentheses, which were ignored, so a
#    question about the planned-treatment cover pinned the emergency-only cover;
#  - a section and its own sub-section were both pinned, spending both slots on one group;
#  - "applicable" in one heading kept it from matching, letting a broader heading win.
_TITLE_STOP_V25 = _TITLE_STOP | {"applicable"}   # "Specific Exclusions applicable to Global Health Cover"

def _core_v25(title: str) -> frozenset:
    title = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", title)
    return frozenset(w for w in _words(title) if w not in _TITLE_STOP_V25)

SECTION_CORES_V25 = {s["section_id"]: _core_v25(s["section_title"]) for s in sections}

def _paren_words(title: str) -> set:
    inner = " ".join(re.findall(r"\(([^)]*)\)", title))
    return {w for w in _words(inner) if w not in _TITLE_STOP}

_SECTION_WORDS = {s["section_id"]: set(_words(s["text"])) for s in sections}
_TITLE_OF = {s["section_id"]: s["section_title"] for s in sections}

def _common_word(w: str, max_sections: int = 3) -> bool:
    return sum(w in ws for ws in _SECTION_WORDS.values()) > max_sections

def title_matches_v25(query: str, max_sections: int = 2):
    q = set(_words(query))
    hits = [sid for sid, core in SECTION_CORES_V25.items() if core and core <= q]
    hits = [a for a in hits if not any(SECTION_CORES_V25[a] < SECTION_CORES_V25[b] for b in hits)]
    # 1. a one-word heading pins only if that word is distinctive (in at most 3 sections)
    hits = [a for a in hits if len(SECTION_CORES_V25[a]) > 1 or not _common_word(next(iter(SECTION_CORES_V25[a])))]
    # 2. headings with the same core words: keep those whose words in parentheses best match the
    #    question; if none of those words appear, keep the first in document order
    by_core = {}
    for a in hits:
        by_core.setdefault(SECTION_CORES_V25[a], []).append(a)
    kept = []
    for same in by_core.values():
        overlap = {a: len(_paren_words(_TITLE_OF[a]) & q) for a in same}
        best = max(overlap.values())
        kept += [a for a in same if overlap[a] == best] if best else same[:1]
    # 3. a sub-section of another hit is already inside that hit's pinned group
    kept = [a for a in kept if not any(_is_child(a, b) for b in kept if b != a)]
    kept.sort(key=lambda sid: -len(SECTION_CORES_V25[sid]))
    return kept[:max_sections]

def title_matches_active(query: str):
    if CONFIG.get("title_pinning_rules", "v2.3") == "v2.5":
        return title_matches_v25(query)
    return title_matches(query)

# v2.6 option (CONFIG["continuation_sections"] = True): sections that only continue the previous one.
# In this policy the Cumulative Bonus notes are numbered as separate sections whose "titles" are
# sentence fragments ("3.9 In case where the Policy", "3.10 If the Sum Insured has been reduced"),
# so title pinning and neighbour expansion never bring them with Section 3.8. A section whose title
# starts like a clause ("In case", "If", "Where", "When", "Provided") and that is not a sub-section of
# the one before it is treated as that section's continuation: it joins its pinned group, it is a
# neighbour for expansion, and its passage label names the section it continues.
_CLAUSE_START = re.compile(r"^(in case|if|where|when|in the event|provided)\b", re.I)

def _find_continuations():
    cont = {}
    for k in range(1, len(sections)):
        s, prev = sections[k], sections[k - 1]
        if _CLAUSE_START.match(s["section_title"]) and not _is_child(s["section_id"], prev["section_id"]):
            cont[s["section_id"]] = cont.get(prev["section_id"], prev["section_id"])
    return cont

CONTINUATION_OF = _find_continuations()

def _family(sid: str) -> str:
    return CONTINUATION_OF.get(sid, sid)

def _continues(a: str, b: str) -> bool:
    """True if section a continues section b (same family, a is a continuation), option on."""
    return bool(CONFIG.get("continuation_sections")) and a != b and a in CONTINUATION_OF \
        and _family(a) == _family(b)

print("Continuation sections:", CONTINUATION_OF or "none",
      "(used when CONFIG['continuation_sections'] = True)")


## 7. LLM

v2.1: the rendered Mistral chat template already starts with `<s>`; v2 tokenized that text with
special tokens again, so every prompt began `<s><s>[INST]`. The text is now tokenized with
`add_special_tokens=False` (one BOS is added only if the template lacks it), and an assertion
fails loudly if a prompt ever starts with two BOS tokens. `use_chat_template=False` reproduces the
original notebook's raw prompt format for the A/B test in Section 10.2. Only newly generated tokens
are decoded in both modes (no prompt echo).

In [ ]:
# ================================================================
# 7) LLM
# ================================================================
from transformers import AutoModelForCausalLM

tokenizer_llm = AutoTokenizer.from_pretrained(CONFIG["llm_model"])
tokenizer_llm.pad_token = tokenizer_llm.eos_token
if CONFIG["llm_load_in_4bit"]:
    try:
        import bitsandbytes  # noqa: F401
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
    from transformers import BitsAndBytesConfig
    model_llm = AutoModelForCausalLM.from_pretrained(
        CONFIG["llm_model"], device_map="auto",
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16))   # T4 has no bfloat16 support
else:
    model_llm = AutoModelForCausalLM.from_pretrained(
        CONFIG["llm_model"], torch_dtype=torch.float16, device_map="auto")
model_llm.eval()

def gpu_report(tag: str = ""):
    if not torch.cuda.is_available():
        return
    free, total = torch.cuda.mem_get_info()
    print(f"[GPU {tag}] in use {(total - free) / 1e9:.1f} / {total / 1e9:.1f} GB "
          f"| free {free / 1e9:.2f} GB")

gpu_report("after loading all models")
_offloaded = [k for k, v in (getattr(model_llm, "hf_device_map", None) or {}).items()
              if v in ("cpu", "disk")]
if _offloaded:
    print(f"WARNING: {len(_offloaded)} LLM modules were placed on CPU/disk because the GPU is full. "
          "Generation will be slow and OOM-prone; set CONFIG['llm_load_in_4bit'] = True.")

# Diagnostic: does the rendered template already carry BOS? (If True, v2's tokenization doubled it.)
_probe = tokenizer_llm.apply_chat_template(
    [{"role": "user", "content": "x"}], tokenize=False, add_generation_prompt=True)
print("Chat template text starts with BOS:", _probe.startswith(tokenizer_llm.bos_token or "<s>"))

def build_llm_inputs(user_msg: str, use_chat_template: bool = None):
    use_tpl = CONFIG["use_chat_template"] if use_chat_template is None else use_chat_template
    bos = tokenizer_llm.bos_token_id
    if use_tpl:
        prompt_text = tokenizer_llm.apply_chat_template(
            [{"role": "user", "content": user_msg}], tokenize=False, add_generation_prompt=True)
        enc_in = tokenizer_llm(prompt_text, return_tensors="pt", add_special_tokens=False)
        if enc_in["input_ids"][0, 0].item() != bos:      # template without BOS -> add exactly one
            enc_in = tokenizer_llm(prompt_text, return_tensors="pt")
    else:
        enc_in = tokenizer_llm(user_msg, return_tensors="pt")   # original notebook's format
    ids = enc_in["input_ids"][0]
    assert not (len(ids) > 1 and ids[0].item() == bos and ids[1].item() == bos), \
        "Prompt starts with two BOS tokens"
    return enc_in.to(model_llm.device)

def generate_answer(user_msg: str, use_chat_template: bool = None) -> str:
    enc_in = build_llm_inputs(user_msg, use_chat_template)
    with torch.no_grad():
        out_ids = model_llm.generate(
            **enc_in,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,
            pad_token_id=tokenizer_llm.eos_token_id,
        )
    return tokenizer_llm.decode(
        out_ids[0][enc_in["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


## 8. LangGraph pipeline

Gate (unchanged): refuse only when the **best** reranker score among the candidates is below −5.0.

v2.3: `retrieve` adds the chunks of title-matched sections to the candidates; `rerank` builds the
context with Section 6.1 and scores any neighbour chunk that was not a candidate (so every source
shows a score). The prompt is the original notebook's instructions plus five scope rules
(`CONFIG["prompt_version"] = "v2.3"`); `"original"` reproduces v2.2's prompt.

In [ ]:
# ================================================================
# 8) LangGraph pipeline
# ================================================================
from langgraph.graph import StateGraph, END

class RAGState(TypedDict):
    query: str
    retrieved: List[dict]
    reranked: List[dict]      # chunks placed in the context, most relevant group first
    scores: List[float]       # reranker score of each of those chunks
    groups: List[List[int]]   # context passages: chunk indices in document order
    top_score: float          # best reranker score among the candidates (used by the gate)
    answer: str
    error: str                # v2.6: set when a node fails (the question is then refused)
    oom_retry: bool           # v2.6: True when the answer came from the shortened out-of-memory retry

_INSTRUCTIONS = """You are a professional insurance policy analyst.

Use ONLY the information provided in the CONTEXT to answer the QUESTION.
Do NOT add, assume, or infer details beyond the given text.
Provide a clear, factual, and complete answer in full sentences.
When relevant, cite the section name or number from the CONTEXT.
"""
# v2.3 scope rules — each targets an error seen in the recorded v2.2 answers
_SCOPE_RULES = """Each CONTEXT passage starts with the section(s) it comes from. A condition, limit or exclusion applies only to the cover named in its own section; do not apply it to a different cover.
Do not present items listed under a different section as part of the cover asked about.
If a passage says a list is provided below, include that list when it appears in the CONTEXT.
When comparing plans, take all figures from the same year's table and name the year.
If the CONTEXT covers only part of the question, answer that part and say what is not shown.
"""
_TAIL = """If the answer cannot be found, reply: “Not specified in the provided policy text.”

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

ORIGINAL_PROMPT = _INSTRUCTIONS + _TAIL          # the original notebook's prompt, verbatim
V23_PROMPT = _INSTRUCTIONS + _SCOPE_RULES + _TAIL

def prompt_template(version: str = None) -> str:
    return V23_PROMPT if (version or CONFIG["prompt_version"]) == "v2.3" else ORIGINAL_PROMPT

def format_groups(groups) -> str:
    return "\n\n".join(f"{group_label(g)}\n{merge_group_text(g)}" for g in groups)

def build_prompt(query: str, groups, version: str = None) -> str:
    return prompt_template(version).format(context=format_groups(groups), query=query.strip())

def retrieve_node(state: RAGState):
    try:
        query = state.get("query", "").strip()
        if not query:
            return {"retrieved": []}
        cands = hybrid_retrieve(query)
        if CONFIG["title_pinning"]:                    # make sure pinned sections get scored
            have = {c["idx"] for c in cands}
            for sid in title_matches_active(query):
                for i in _pinned_group(sid):
                    if i not in have:
                        cands.append(final_chunks[i]); have.add(i)
        return {"retrieved": cands}
    except Exception as e:
        print(f"[retrieve] failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        return {"retrieved": [], "error": f"retrieve: {type(e).__name__}: {e}"}

def rerank_node(state: RAGState):
    empty = {"reranked": [], "scores": [], "groups": [], "top_score": float("-inf")}
    try:
        retrieved = state.get("retrieved", [])
        if not retrieved:
            return empty
        q = state["query"].strip()
        _, scores = rerank_all(q, [r["text"] for r in retrieved])
        groups, score_of = build_context(q, retrieved, scores)
        kept = [i for g in groups for i in g]
        missing = [i for i in kept if i not in score_of]        # neighbours outside the candidates
        if missing:
            _, extra = rerank_all(q, [final_chunks[i]["text"] for i in missing])
            score_of.update({i: float(s) for i, s in zip(missing, extra)})
        free_gpu()   # v2.2: hand reranker memory back before generation
        return {"reranked": [final_chunks[i] for i in kept],
                "scores": [score_of[i] for i in kept],
                "groups": groups,
                "top_score": float(max(scores)) if len(scores) else float("-inf")}
    except Exception as e:
        print(f"[rerank] failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        return {**empty, "error": f"rerank: {type(e).__name__}: {e}"}

def gate(state: RAGState) -> str:
    top = state.get("top_score", float("-inf"))
    if state.get("groups") and top >= CONFIG["rerank_score_threshold"]:
        return "answer"
    return "no_answer"

def answer_node(state: RAGState):
    try:
        groups = state.get("groups", [])
        if not groups:
            return {"answer": NO_ANSWER_TEXT}
        try:
            return {"answer": generate_answer(build_prompt(state["query"], groups))}
        except torch.cuda.OutOfMemoryError:
            # v2.2: free memory and retry once with the most relevant passages only
            free_gpu()
            limit, keep, used = CONFIG["oom_retry_context_tokens"], [], 0
            for g in groups:                       # most relevant group first
                part = []
                for i in g:
                    t = block_tokens(final_chunks[i])
                    if used + t > limit and (keep or part):
                        break
                    part.append(i); used += t
                if part:
                    keep.append(part)
                if used >= limit:
                    break
            kept = [i for g in keep for i in g]
            print(f"[answer] CUDA out of memory — retrying with {len(kept)} of "
                  f"{sum(len(g) for g in groups)} chunks ({used} tokens)")
            gpu_report("before retry")
            ans = generate_answer(build_prompt(state["query"], keep))
            score_of = {c["idx"]: s for c, s in zip(state["reranked"], state["scores"])}
            return {"answer": ans, "groups": keep, "oom_retry": True,
                    "reranked": [final_chunks[i] for i in kept], "scores": [score_of[i] for i in kept]}
    except Exception as e:
        print(f"[answer] failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        return {"answer": NO_ANSWER_TEXT, "error": f"answer: {type(e).__name__}: {e}"}
    finally:
        free_gpu()

def no_answer_node(state: RAGState):
    return {"answer": NO_ANSWER_TEXT}

workflow = StateGraph(RAGState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("rerank", rerank_node)
workflow.add_node("answer", answer_node)
workflow.add_node("no_answer", no_answer_node)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "rerank")
workflow.add_conditional_edges("rerank", gate,
                               {"answer": "answer", "no_answer": "no_answer"})
workflow.add_edge("answer", END)
workflow.add_edge("no_answer", END)

app = workflow.compile()
print("Pipeline ready: retrieve (+ title matches) -> rerank -> context groups -> gate -> answer | no_answer")


## 9. Ask questions

In [ ]:
# ================================================================
# 9) Query helper — shows sources + scores for auditing
# ================================================================
def ask(query: str, show_sources: bool = True):
    result = app.invoke({"query": query})
    print("Q:", query.strip())
    if show_sources and result.get("reranked"):
        srcs = [f"{c['section_id']} ({s:.2f})"
                for c, s in zip(result["reranked"], result.get("scores", []))]
        print("Sources:", ", ".join(srcs))
        print("Context tokens:", sum(c["token_count"] for c in result["reranked"]))
    print("\nA:", result["answer"])
    print("=" * 80)
    return result

_ = ask("What expenses are covered under the Overseas Travel Secure option? Give complete details.")


In [ ]:
# The full original query set
_ = ask("What is the difference between Optima Secure and Optima Super Secure plans? Give complete details.")
_ = ask("What are the standard exclusions under the my:Optima Secure policy? Give complete details.")
_ = ask("What are the exact specific conditions with a 24-month waiting period? Give complete details.")
_ = ask("Under what circumstances can obesity-related surgery be covered under Excl06, and what certifications are required?")
_ = ask("How to file reimbursement claims? Give me step wise complete procedure.")
_ = ask("What is the procedure for cashless claims in India? Give me step wise complete procedure.")
_ = ask("What is the Plus Benefit? Explain in detail.")
_ = ask("What are the exact inclusions for pre-hospitalization expenses? Give complete details.")
_ = ask("What are the exact inclusions for post-hospitalization expenses? Give complete details.")


## 10. Evaluation harness

> **v2.5 note:** 3 of the 9 answerable `GOLD` questions expect keywords that are not in their gold
> sections: "incurred and paid overseas" comes from 4.10.1, not 4.11; "2 years" and "irrespective of
> a claim" come from the introduction, not 4.4; and Excl03 is in 10.1.3, which the exclusions question
> does not list. `GOLD` is kept unchanged for continuity; the test set in Section 10.3 corrects these
> and is the one to use for decisions.

Section IDs in `GOLD` were verified against the section ladder this parser produces on your policy
document. Run after **every** change to chunking, retrieval, thresholds, or prompts.

In [ ]:
# ================================================================
# 10) Gold-set evaluation
# ================================================================
GOLD = [
    {
        "query": "What expenses are covered under the Overseas Travel Secure option? Give complete details.",
        "gold_sections": {"4.11"},
        "must_contain": ["5 consecutive days", "accompanying person", "incurred and paid overseas"],
    },
    {
        "query": "What is the Plus Benefit? Explain in detail.",
        "gold_sections": {"4.4", "4.4.1"},
        "must_contain": ["100%", "2 years", "irrespective of a claim"],
    },
    {
        "query": "What are the exact inclusions for pre-hospitalization expenses?",
        "gold_sections": {"3.5"},
        "must_contain": ["60 days", "prior"],
    },
    {
        "query": "What are the exact inclusions for post-hospitalization expenses?",
        "gold_sections": {"3.6"},
        "must_contain": ["180 days", "discharge"],
    },
    {
        "query": "How to file reimbursement claims? Give me step wise complete procedure.",
        "gold_sections": {"11.6", "11.7"},
        "must_contain": ["30 days from the date of discharge", "claim form"],
    },
    {
        "query": "What is the procedure for cashless claims in India? Give me step wise complete procedure.",
        "gold_sections": {"11.2"},
        "must_contain": ["Network Provider", "pre-authorization"],
    },
    {   # v2.3: the list lives in 10.1.1, which v2.2 never retrieved
        "query": "What are the exact specific conditions with a 24-month waiting period? Give complete details.",
        "gold_sections": {"10.1", "10.1.1"},
        "must_contain": ["24 months", "Non-infective Arthritis"],
    },
    {
        "query": "What are the standard exclusions under the my:Optima Secure policy? Give complete details.",
        "gold_sections": {"10.1"},
        "must_contain": ["Excl01", "Excl02", "Excl03"],
    },
    {
        "query": "What is the difference between Optima Secure and Optima Super Secure plans? Give complete details.",
        "gold_sections": {"2.1", "2.2", "2.3"},
        "must_contain": ["₹20 lakh", "Year"],
    },
    # Negative control — should route to no_answer:
    {
        "query": "What is the coverage for pet insurance under this policy?",
        "gold_sections": set(),
        "must_contain": [NO_ANSWER_TEXT],
    },
]

def evaluate(gold=GOLD):
    rows = []
    for case in gold:
        result = app.invoke({"query": case["query"]})
        got_secs = {c["section_id"] for c in result.get("reranked", [])}
        if case["gold_sections"]:
            recall = len(got_secs & case["gold_sections"]) / len(case["gold_sections"])
        else:
            recall = None
        ans = result["answer"].lower()
        hits = sum(kw.lower() in ans for kw in case["must_contain"])
        rows.append({
            "query": case["query"][:55] + "...",
            "section_recall": recall if recall is None else round(recall, 2),
            "keyword_hits": f"{hits}/{len(case['must_contain'])}",
            "routed": "no_answer" if is_no_answer(result["answer"]) else "answer",
            "top_score": round(result["top_score"], 2) if result.get("scores") else None,
            "context_tokens": sum(c["token_count"] for c in result.get("reranked", [])),
        })
    return pd.DataFrame(rows)

gold_df = evaluate()   # kept for the results zip (Section 12)
gold_df


In [ ]:
# ================================================================
# 10.1) Threshold sweep — tune the gate against the gold set
# ================================================================
def sweep_thresholds(values=(-8, -6, -5, -4, -3, -2, -1, 0)):
    original = CONFIG["rerank_score_threshold"]
    rows = []
    for thr in values:
        CONFIG["rerank_score_threshold"] = thr
        df = evaluate()
        answered = (df["routed"] == "answer").sum()
        neg_ok = df.iloc[-1]["routed"] == "no_answer"   # negative control refused?
        rows.append({"threshold": thr, "answered": f"{answered}/{len(df)}",
                     "negative_control_refused": neg_ok})
    CONFIG["rerank_score_threshold"] = original
    return pd.DataFrame(rows)

# Pick the HIGHEST threshold that still answers all answerable queries
# while refusing the negative control.
# sweep_thresholds()


## 10.1 Regression check against the original notebook

`ORIGINAL_EVIDENCE` holds, for each of the original notebook's questions, the exact passages its
answers drew on (extracted by matching each original answer against the context the original
pipeline printed). Each passage is tagged with its section and whether that section is on-topic
for the question.

For every question this reports:
- **evidence_recall_all** — share of the original answer's evidence that is present in the v2.1
  context. The original pipeline scores 1.00 by construction. Target: ≥ 0.90.
- **evidence_recall_on_topic** — same, restricted to on-topic sections.
- **answer_grounding** — share of the new answer's wording found in its context (higher = less
  invented text). `orig_answer_grounding` is the original answer's score against its own context.

Read the last two columns together: several original answers were long but weakly grounded
(standard exclusions 9%, plan comparison 36%) — a shorter answer there is not a regression.

In [ ]:
# ================================================================
# 10.1) Regression check vs the original notebook's answers
# ================================================================
ORIGINAL_EVIDENCE = json.loads(r"""{
 "What is the difference between Optima Secure and Optima Super Secure plans? Give complete details.": {
  "passages": [
   {
    "section": "0",
    "on_topic": true,
    "text": "expenses incurred on hospitalization due to Illness or Accident."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "offers additional coverage amount equivalent to 100%/200% of the Base Sum Insured."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "offers additional coverage equivalent to 100% of the Base Sum Insured in 2 years irrespective of a claim."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "restores 100% of Base Sum Insured automatically on partial or complete utilization of Sum Insured"
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "pays towards the Non-Medical expenses like gloves, food charges and other consumables during hospitalization."
   },
   {
    "section": "2.2",
    "on_topic": true,
    "text": "Maximum permissible amount for a single hospitalization claim:"
   },
   {
    "section": "2.2",
    "on_topic": true,
    "text": "Maximum permissible amount for all hospitalization claims in a policy year:"
   },
   {
    "section": "2.3",
    "on_topic": true,
    "text": "Maximum permissible amount for a single hospitalization claim:"
   },
   {
    "section": "2.3",
    "on_topic": true,
    "text": "Maximum permissible amount for all hospitalization claims in a policy year:"
   },
   {
    "section": "13.2",
    "on_topic": false,
    "text": "the base sum insured. The"
   }
  ],
  "orig_answer_grounding": 0.364
 },
 "What are the standard exclusions under the my:Optima Secure policy? Give complete details.": {
  "passages": [
   {
    "section": "10.1",
    "on_topic": true,
    "text": "All the Waiting Periods and exclusions listed below shall be applicable individually for each Insured Person and claims shall be assessed accordingly. a."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of a pre-existing disease (PED) and its direct complications shall be excluded until the expiry of 36 months of continuous coverage after the date of inception of the first policy with"
   }
  ],
  "orig_answer_grounding": 0.094
 },
 "What are the exact standard exclusions under the my:Optima Secure policy? Give complete details.": {
  "passages": [
   {
    "section": "10.1",
    "on_topic": true,
    "text": "All the Waiting Periods and exclusions listed below shall be applicable individually for each Insured Person and claims shall be assessed accordingly. a."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of a pre-existing disease (PED) and its direct complications shall be excluded until the expiry of 36 months of continuous coverage after the date of inception of the first policy with"
   }
  ],
  "orig_answer_grounding": 0.088
 },
 "What are the exact specific conditions with a 24-month waiting period? Give complete details.": {
  "passages": [
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "shall be excluded until the expiry of"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "months of continuous coverage after the date of inception of the first policy with"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "If the Insured Person is continuously covered without any break as defined under the"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "then waiting period for the same would be reduced to the extent of prior coverage."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of the listed Conditions, surgeries/treatments shall be excluded until the expiry of 24 months of continuous coverage after the date of inception of the first Policy with"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "This exclusion shall not be applicable for claims arising due to an Accident."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "any of the specified disease/procedure falls under the waiting period specified for Pre-Existing diseases, then the longer of the two waiting periods shall apply."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "The waiting period for listed conditions shall apply even if contracted after the Policy or declared and accepted without a specific exclusion."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "If the Insured Person is continuously covered without any break as defined under the applicable norms on portability stipulated by IRDAI, then waiting period for the same would be reduced to the extent of prior coverage."
   },
   {
    "section": "10.1.1",
    "on_topic": true,
    "text": "Joint Conditions: Non-infective Arthritis, Pilonidal Sinus, Osteoarthritis, Osteoporosis, Gout, Rheumatism. Gastrointestinal"
   },
   {
    "section": "10.1.1",
    "on_topic": true,
    "text": "Anorectal Conditions: Diseases of Gall Bladder including Cholecystitis, Pancreatitis, Ulcer and Erosion of Stomach and Duodenum, All Forms of Cirrhosis, Gastro Esophageal Reflux Disorder (GERD), Fissure"
   },
   {
    "section": "10.1.1",
    "on_topic": true,
    "text": "Fistula in Anus, Haemorrhoids, Perineal Abscesses, Perianal Abscesses."
   }
  ],
  "orig_answer_grounding": 0.469
 },
 "How to file reimbursement claims? Give me step wise complete procedure.": {
  "passages": [
   {
    "section": "11.3",
    "on_topic": false,
    "text": "as specified in the Policy,"
   },
   {
    "section": "11.5",
    "on_topic": false,
    "text": "days from the date of"
   },
   {
    "section": "11.6",
    "on_topic": true,
    "text": "submit the necessary documents to the Company within the prescribed time limit"
   },
   {
    "section": "11.6",
    "on_topic": true,
    "text": "Claim Types and Prescribed Time Limits: Reimbursement of Hospitalization, Day Care Treatment, or Pre-Hospitalization Expenses: Must be submitted within 30 days from the date of discharge from the hospital. Reimbursement of Post-Hospitalization Expenses: Must be submitted within 15 days from the completion of post-hospitalization treatment."
   },
   {
    "section": "11.7",
    "on_topic": true,
    "text": "The reimbursement claim is to be supported with the following documents"
   },
   {
    "section": "11.7",
    "on_topic": true,
    "text": "within the prescribed time limit. a. Duly Completed claim form, b. Photo ID and Age Proof, c. Copy of the Hospital's Registration Certificate/Hospital Registration number in case of Hospitalization in any non Network Provider of the Company or certificate from Hospital authorities providing facilities available including number of beds, d. Discharge Card"
   },
   {
    "section": "11.7",
    "on_topic": true,
    "text": "Transfer Summary, e. Final Hospital bill with all original deposit and final payment receipt and refund receipt(s), if advance amount refunded, f. Invoice with payment receipt and implant stickers for all implants used during Surgeries e.g. lens sticker and invoice in cataract Surgery, stent invoice and sticker in Angioplasty Surgery, g. All previous consultation papers indicating history and treatment details for current."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "for claims outside India of Insured Person and Accompanying Person (as applicable)"
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Passport copy with entry and exit stamps"
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Flight Tickets and Boarding Pass, if applicable"
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Written advice from the overseas treating Medical Practitioner for requirement of an accompanying person during treatment."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Any other relevant document required by Company for assessment of the claim."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "The Company shall only accept bills/invoices/medical treatment related documents only in the Insured Person's name for whom the claim is submitted."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "In the event of a claim lodged under the Policy and the original documents having been submitted to any other insurer, the Company shall accept the copy of the documents and claim settlement advice, duly certified by the other insurer subject to satisfaction of the Company."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "If requested by the Company, at the Company's cost, the Insured Person must submit to medical examination by Medical Practitioner appointed by the Company as often as it is considered reasonable and necessary and Company's representatives must be permitted to inspect the medical and Hospitalization records pertaining to the Insured Person's treatment, and to investigate the circumstances pertaining to the claim."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Any delay in notification or submission may be condoned on merit where delay is proved to be for reasons beyond the control of the Insured Person. Section"
   },
   {
    "section": "12.2",
    "on_topic": false,
    "text": "within 30 days from the date of"
   }
  ],
  "orig_answer_grounding": 0.893
 },
 "What is the procedure for cashless claims in India? Give me step wise complete procedure.": {
  "passages": [
   {
    "section": "11.2",
    "on_topic": true,
    "text": "Procedure for Cashless Claims In India"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "Cashless request form is available with the Network Provider."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "The Network Provider shall obtain"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "relevant information from the Insured"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "and send a Cashless Facility request to the Company for authorization."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "cashless request form and related medical information from the Insured"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "pre-authorization letter to the Network Provider after verification."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "At the time of discharge, the Insured Person shall verify and sign the discharge papers along with final bill, pay for non-medical and inadmissible expenses."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "The Company reserves the right to deny pre-authorization in case the Insured Person is unable to provide the relevant medical details."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "the Insured Person may obtain"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "treating doctor's advice and submit the claim documents to the Company for reimbursement."
   },
   {
    "section": "11.3",
    "on_topic": false,
    "text": "as specified in the Policy,"
   },
   {
    "section": "11.4",
    "on_topic": false,
    "text": "Procedure for Cashless Claims in"
   },
   {
    "section": "11.5",
    "on_topic": true,
    "text": "The Company reserves the right to"
   },
   {
    "section": "11.5",
    "on_topic": true,
    "text": "Pre-authorization issued by the Company shall be valid for 15 days from the date of issuance (or expiry of the Policy, whichever is earlier)."
   },
   {
    "section": "11.5",
    "on_topic": true,
    "text": "The Company shall make payment for the Cashless facility to the authorized amount, directly to the Network Provider."
   },
   {
    "section": "11.6",
    "on_topic": false,
    "text": "days from the date of"
   }
  ],
  "orig_answer_grounding": 0.743
 },
 "What is the Plus Benefit? Explain in detail.": {
  "passages": [
   {
    "section": "0",
    "on_topic": true,
    "text": "additional coverage amount equivalent to"
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "of the Base Sum Insured."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "equivalent to 100% of the Base Sum Insured in"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "the Policy is issued on an individual basis, the Plus Benefit shall be added and available individually to the Insured Person."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "floater basis, the Plus Benefit shall be added and available to all Family Members on a floater basis."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "if the Policy is renewed and due premium is received within the Grace Period."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "Renewed on a floater policy basis"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "the Plus Benefit to be carried forward for credit in"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "Renewed Policy shall be the lowest one that is applicable among all the Insured Persons."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "in cases where the Policy is split due to"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "child attaining the Age of 25 years, the Plus Benefit of the expiring policy shall be apportioned to such Renewed Policies in the proportion of the Sum Insured of each Renewed Policy"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "If the Sum Insured has been reduced at the time of Renewal, the applicable Plus Benefit shall be reduced in the same proportion to the Sum Insured in"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "If the Sum Insured under the Policy has been increased at the time of Renewal, the Plus Benefit shall be calculated on the Sum Insured of the last completed Policy Year."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "If the Policy Period is of two or three years, the Plus Benefit shall be credited post completion of each Policy Year, and will be available for any claims made in the subsequent Policy Year."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "added to the Policy during subsequent Renewals will be eligible for the Plus Benefit as per their Renewal terms."
   }
  ],
  "orig_answer_grounding": 0.778
 },
 "What are the exact inclusions for pre-hospitalization expenses? Give complete details.": {
  "passages": [
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Medicines and drugs prescribed in writing by"
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Intravenous fluids, blood transfusion, surgical appliances, allowable consumables and/or enteral feedings."
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "The cost of prosthetics and other devices or equipment, if implanted internally during Surgery."
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Expenses incurred on road Ambulance if the Insured Person is required to be transferred to the nearest Hospital for Emergency Care or from one Hospital to another Hospital or from Hospital to Home (within"
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Care Dental Treatment, necessitated due to disease or Injury"
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Plastic surgery, necessitated due to Injury"
   },
   {
    "section": "3.4",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person"
   },
   {
    "section": "3.5",
    "on_topic": true,
    "text": "Pre-Hospitalization Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to 60 days immediately prior to the date of admissible Hospitalization covered under the Policy."
   },
   {
    "section": "3.6",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "The organ donor's Pre-Hospitalization Expenses and Post Hospitalization Expenses."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Expenses related to organ transportation or preservation."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Any other Medical Expenses or Hospitalization consequent to the organ harvesting."
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "if the Insured Person is"
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "under Section 3.1 (Hospitalization Expenses)."
   },
   {
    "section": "10.2",
    "on_topic": false,
    "text": "laundry charges, attendant charges, ambulance collar, ambulance equipment, baby food, baby utility charges"
   }
  ],
  "orig_answer_grounding": 0.653
 },
 "What are the exact inclusions for post-hospitalization expenses? Give complete details.": {
  "passages": [
   {
    "section": "3.1",
    "on_topic": false,
    "text": "a. Room Rent, boarding, nursing expenses as provided by the Hospital"
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Nursing Home. b. Intensive Care Unit (ICU)"
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Intensive Cardiac Care Unit (ICCU) expenses. c. Surgeon, anaesthetist, Medical Practitioner, consultants, specialist Fees during Hospitalization forming part of Hospital bill. d. Investigative treatments and diagnostic procedures directly related to Hospitalization. e. Medicines and drugs prescribed in writing by Medical Practitioner. f. Intravenous fluids, blood transfusion, surgical appliances, allowable consumables and/or enteral feedings."
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "The cost of prosthetics and other devices or equipment, if implanted internally during Surgery."
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "i. Expenses incurred on road Ambulance if the Insured Person is required to be transferred to the nearest Hospital for Emergency Care or from one Hospital to another Hospital or from Hospital to Home (within same city) following Hospitalization."
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "In patient Care Dental Treatment, necessitated due to disease or Injury"
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Plastic surgery, necessitated due to Injury"
   },
   {
    "section": "3.4",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person"
   },
   {
    "section": "3.5",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to"
   },
   {
    "section": "3.6",
    "on_topic": true,
    "text": "Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to 180 days from the date of discharge from the Hospital,"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "incurred by the Insured Person"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "under Section 3.1 (Hospitalization Expenses)"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "The organ donor's Pre-Hospitalization Expenses and Post Hospitalization Expenses."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Expenses related to organ transportation or preservation."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Any other Medical Expenses or Hospitalization consequent to the organ harvesting."
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "if the Insured Person is"
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "under Section 3.1 (Hospitalization Expenses)."
   },
   {
    "section": "4.3",
    "on_topic": false,
    "text": "Non-Medical Expenses listed under Annexure B to this Policy incurred in relation to a claim admissible under Section 3 (Base Coverage) during the Policy Year. Exclusion (k) of Section 10.2"
   },
   {
    "section": "4.3",
    "on_topic": false,
    "text": "Specific Exclusions shall not apply to this Cover."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "The provision or fitting of hearing aids, spectacles or contact lenses."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "expenses for alopecia, baldness including corticosteroids and topical immunotherapy wigs, toupees, hair pieces, any non-surgical hair replacement methods, optometric therapy."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Artificial limbs and/or device used for diagnosis or treatment (except when used intra-operatively), prosthesis, corrective devices external durable medical equipment of any kind, wheelchairs, crutches, and oxygen concentrator for bronchial asthma/ COPD conditions, cost of cochlear implant(s) unless necessitated by an Accident."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any treatment or part of a treatment that is not of a reasonable charge and not Medically Necessary. Drugs or treatments which are not supported by a prescription."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any permanent exclusion applied on any medical or physical condition or treatment of an Insured Person as specifically mentioned in the Policy Schedule and as specifically accepted by Policyholder/Insured Person."
   }
  ],
  "orig_answer_grounding": 0.808
 },
 "What expenses are covered under the Overseas Travel Secure option? Give complete details.": {
  "passages": [
   {
    "section": "4.7",
    "on_topic": false,
    "text": "Section 4.9 (Global Health Cover (Emergency Treatments Only)),"
   },
   {
    "section": "4.10.1",
    "on_topic": false,
    "text": "Only those Pre-hospitalization Medical Expenses and/or Post-hospitalization Medical Expenses shall be admissible under this benefit that have been incurred and paid overseas. Such expenses should be related to an admissible overseas Hospitalization claim only (as per details in invoice/supporting documents)."
   },
   {
    "section": "4.11",
    "on_topic": true,
    "text": "conditions are fulfilled: a. The overseas treating Medical Practitioner has advised a minimum hospitalization of 5 consecutive days and has also advised the requirement of an accompanying person during treatment. b. We have accepted a claim under"
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Treatment taken on outpatient basis."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "The provision or fitting of hearing aids, spectacles or contact lenses."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any treatment and associated expenses for alopecia, baldness"
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "corticosteroids and topical immunotherapy wigs, toupees, hair pieces, any non-surgical hair replacement methods, optometric therapy."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Expenses for Artificial limbs and/or device used for diagnosis or treatment (except when used intra-operatively), prosthesis, corrective devices external durable medical equipment of any kind, wheelchairs, crutches, and oxygen concentrator for bronchial asthma/ COPD conditions, cost of cochlear implant(s) unless necessitated by an Accident."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any treatment or part of a treatment that is not of a reasonable charge and not Medically Necessary. Drugs or treatments which are not supported by a prescription."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any permanent exclusion applied on any medical or physical condition or treatment of an Insured Person as specifically mentioned in the Policy Schedule and as specifically accepted by Policyholder/Insured Person. Such exclusions"
   }
  ],
  "orig_answer_grounding": 0.832
 }
}""")

def _key(w: str) -> str:
    return re.sub(r"[^\w₹%]", "", w.lower())

def _keys(text: str):
    return [k for k in (_key(w) for w in text.split()) if k]

def _grams(keys, n=5):
    return {tuple(keys[i:i + n]) for i in range(len(keys) - n + 1)}

def _covered(keys, grams, n=5):
    cov = [False] * len(keys)
    for i in range(len(keys) - n + 1):
        if tuple(keys[i:i + n]) in grams:
            for j in range(i, i + n):
                cov[j] = True
    return sum(cov), len(keys)

def evidence_recall(text: str, passages, n=5):
    g = _grams(_keys(text), n)
    hit = tot = 0
    for p in passages:
        h, t = _covered(_keys(p["text"]), g, n)
        hit, tot = hit + h, tot + t
    return round(hit / tot, 2) if tot else None

def grounding(answer: str, context: str, n=5):
    h, t = _covered(_keys(answer), _grams(_keys(context), n), n)
    return round(h / t, 2) if t else None

def regression_check():
    rows = []
    for q, fx in ORIGINAL_EVIDENCE.items():
        res = app.invoke({"query": q})
        ctx = "\n\n".join(c["text"] for c in res.get("reranked", []))
        on = [p for p in fx["passages"] if p["on_topic"]]
        rows.append({
            "query": q[:50] + "...",
            "context_tokens": sum(c["token_count"] for c in res.get("reranked", [])),
            "evidence_recall_all": evidence_recall(ctx, fx["passages"]),
            "evidence_recall_on_topic": evidence_recall(ctx, on) if on else None,
            "answer_grounding": grounding(res["answer"], ctx),
            "orig_answer_grounding": fx["orig_answer_grounding"],
        })
    return pd.DataFrame(rows)

regression_df = regression_check()   # kept for the results zip (Section 12)
regression_df


## 10.2 Generation A/B on identical contexts (optional, ~3× the LLM calls)

Retrieval runs once per question; the same v2.3 context is then answered three ways:
- **A_original** — original prompt, raw text, no chat template (the original notebook's generation)
- **B_v2.2_prompt** — original instructions + section labels + chat template (v2.2's prompt)
- **C_v2.3_rules** — B plus the five scope rules (v2.3 default)

`answer_evidence_recall` = how much of the original answer's evidence each new answer reproduces
(completeness); `grounding` = share of the answer found in the context (faithfulness). Keep
`CONFIG["prompt_version"] = "v2.3"` only if C is at least as complete as B with equal or better
grounding.

In [ ]:
# ================================================================
# 10.2) Generation A/B — same retrieved context, three prompt variants
# ================================================================
def generation_ab(questions=None):
    rows = []
    for q in (questions or list(ORIGINAL_EVIDENCE)):
        st = retrieve_node({"query": q}); st["query"] = q
        st.update(rerank_node(st))
        groups = st["groups"]
        ctx_plain = "\n\n".join(merge_group_text(g) for g in groups)
        passages = ORIGINAL_EVIDENCE.get(q.strip(), {}).get("passages", [])
        variants = {
            "A_original": (ORIGINAL_PROMPT.format(context=ctx_plain, query=q.strip()), False),
            "B_v2.2_prompt": (build_prompt(q, groups, "original"), True),
            "C_v2.3_rules": (build_prompt(q, groups, "v2.3"), True),
        }
        for name, (prompt, tpl) in variants.items():
            ans = generate_answer(prompt, use_chat_template=tpl)
            rows.append({"query": q[:40] + "...", "variant": name,
                         "answer_words": len(ans.split()),
                         "answer_evidence_recall": evidence_recall(ans, passages) if passages else None,
                         "grounding": grounding(ans, ctx_plain)})
    return pd.DataFrame(rows)

# ab = generation_ab(); ab.pivot(index="query", columns="variant",
#                                values=["answer_evidence_recall", "grounding"])


## 10.3 Test set (94 questions)

Written from the policy text to cover the kinds of questions people ask. The first 10 are the
original gold questions, with corrected key facts.

| Type | Questions | Example |
|---|---|---|
| fact | 22 | What is the waiting period for pre-existing diseases? |
| everyday wording | 15 | Will the policy pay for my wife's delivery? |
| list | 13 | Which expenses are covered under Hospitalization Expenses? |
| conditional | 12 | Is Daily Cash for Shared Room paid for time spent in the ICU? |
| definition | 8 | What is the Automatic Restore Benefit? |
| out of scope | 7 | How much does the policy pay for lost baggage during travel? |
| procedure | 5 | What is the procedure for cashless claims outside India? |
| comparison | 5 | How do the Cumulative Bonus and the Plus Benefit differ? |
| several sections | 4 | Which covers are not subject to the Aggregate Deductible? |
| false premise | 3 | What is the room rent limit of 1% of the Sum Insured per day? |

24 questions (`paraphrase=True`) avoid the words of the section heading, so they test retrieval beyond
title matching. K11 and K12 ask about the Cumulative Bonus notes, which sit in sections whose labels
are sentence fragments ("3.9 In case where the Policy"); they show whether such sections are found.
Each question has its gold sections, its key facts (each with alternative phrasings) and a short
reference answer. `check_testset()` re-checks everything against the policy text loaded in Section 1.

In [ ]:
# ================================================================
# Test set v2 — my:Optima Secure policy (questions, gold evidence, key facts, references)
# ================================================================
# Each item:
#   id, type, question, answerable, paraphrase (True = worded without the section heading's words)
#   gold      : section ids that hold the evidence
#   facts     : key facts; each fact is a list of alternative phrasings, any one counts.
#               Matching is case-insensitive after normalisation (₹/Rs. dropped, hyphens -> spaces,
#               "-isation" -> "-ization"). "re:" marks a regular expression.
#   reference : a short correct answer written from the policy text (for reading results)
# Every fact phrase was checked to occur in the gold sections; every out-of-scope question was
# checked to have its key terms absent from the policy.
TESTSET = [
    # ---------------- the 10 original gold questions (kept verbatim for continuity) ----------------
    {"id": "G01", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What expenses are covered under the Overseas Travel Secure option? Give complete details.",
     "gold": ["4.11"],
     "facts": [["5 consecutive days"], ["accompanying person"], ["economy"], ["15,000"], ["30 days"],
               ["meals, laundry", "meals"]],
     "reference": "Overseas Travel Secure (4.11), available only with the Global or Global Plus plan, pays economy airfare for the insured person and one accompanying person and accommodation for the accompanying person up to Rs. 15,000 per day for up to 30 days in a policy year, if the overseas doctor advised at least 5 consecutive days of hospitalization and a Global Health Cover claim was accepted; meals, laundry and other transport are not paid."},
    {"id": "G02", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What is the Plus Benefit? Explain in detail.",
     "gold": ["4.4", "4.4.1"],
     "facts": [["50%"], ["100%"], ["irrespective"], ["grace period"]],
     "reference": "On each renewal without a break, 50% of the expiring Base Sum Insured is added, accumulating up to 100% of the Base Sum Insured irrespective of claims; it needs renewal within the Grace Period and can be used only for Section 3 and Section 4.3 claims."},
    {"id": "G03", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What are the exact inclusions for pre-hospitalization expenses?",
     "gold": ["3.5"],
     "facts": [["60 days"], ["3.1", "admissible hospitalization"]],
     "reference": "Pre-hospitalization medical expenses are covered for up to 60 days immediately before an admissible hospitalization under Section 3.1."},
    {"id": "G04", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What are the exact inclusions for post-hospitalization expenses?",
     "gold": ["3.6"],
     "facts": [["180 days"], ["discharge"]],
     "reference": "Post-hospitalization medical expenses are covered for up to 180 days from discharge, following an admissible hospitalization claim under Section 3.1."},
    {"id": "G05", "type": "procedure", "paraphrase": False, "answerable": True,
     "question": "How to file reimbursement claims? Give me step wise complete procedure.",
     "gold": ["11.6", "11.7"],
     "facts": [["30 days"], ["15 days"], ["claim form"], ["discharge"]],
     "reference": "Submit the claim documents within 30 days of discharge (hospitalization, day care, pre-hospitalization) or within 15 days of finishing post-hospitalization treatment, with the claim form, photo ID, discharge summary, final bill and the other documents in Section 11.7."},
    {"id": "G06", "type": "procedure", "paraphrase": False, "answerable": True,
     "question": "What is the procedure for cashless claims in India? Give me step wise complete procedure.",
     "gold": ["11.2"],
     "facts": [["network provider"], ["pre authorization", "preauthorization"], ["discharge"], ["reimbursement"]],
     "reference": "Treatment at a Network Provider subject to pre-authorization: the provider sends the cashless request, the Company issues a pre-authorization letter, the insured signs the discharge papers and pays non-medical expenses, and if cashless is denied the claim can be filed for reimbursement."},
    {"id": "G07", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What are the exact specific conditions with a 24-month waiting period? Give complete details.",
     "gold": ["10.1", "10.1.1", "10.1.2"],
     "facts": [["24 months"], ["non infective arthritis"], ["cataract"], ["hernia"], ["accident"]],
     "reference": "Under Excl02 the listed conditions (such as non-infective arthritis, gall bladder disease, kidney stones, cataract) and procedures (such as hernia repair, joint replacement) are excluded for 24 months of continuous coverage, except claims arising from an accident."},
    {"id": "G08", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What are the standard exclusions under the my:Optima Secure policy? Give complete details.",
     "gold": ["10.1", "10.1.3", "10.1.4", "10.1.5"],
     "facts": [["excl01", "pre existing"], ["excl02", "24 months"], ["excl03", "30 day"], ["excl06", "obesity"],
               ["excl09", "hazardous"], ["excl18", "maternity"]],
     "reference": "Standard exclusions Excl01 to Excl18: pre-existing diseases (36 months), specified diseases (24 months), 30-day waiting period, investigation, rest cure, obesity surgery, gender change, cosmetic surgery, hazardous sports, breach of law, excluded providers, addiction, spas, supplements, refractive error, unproven treatment, infertility and maternity."},
    {"id": "G09", "type": "comparison", "paraphrase": False, "answerable": True,
     "question": "What is the difference between Optima Secure and Optima Super Secure plans? Give complete details.",
     "gold": ["2.1", "2.2", "2.3"],
     "facts": [["secure benefit"], ["20 lakh"], ["30 lakh"], ["40 lakh"]],
     "reference": "With a Rs. 10 lakh Base Sum Insured, the Secure Benefit is Rs. 10 lakh for Optima Secure and Rs. 20 lakh for Super Secure, so Super Secure allows Rs. 10 lakh more: in Year 1 the single-claim maximum is Rs. 20 lakh vs Rs. 30 lakh and the yearly maximum Rs. 30 lakh vs Rs. 40 lakh."},
    {"id": "G10", "type": "out_of_scope", "paraphrase": False, "answerable": False,
     "question": "What is the coverage for pet insurance under this policy?",
     "gold": [], "facts": [], "absent_terms": ["pet", "pets", "animal insurance"],
     "reference": "Not specified in the provided policy text."},

    # ---------------- single facts ----------------
    {"id": "F01", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the age range of people covered under this policy?",
     "gold": ["1"], "facts": [["91 days"], ["65 years"]],
     "reference": "The policy covers insured persons aged 91 days to 65 years."},
    {"id": "F02", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the maximum entry age for a dependent child?",
     "gold": ["1"], "facts": [["25 years"]],
     "reference": "A dependent child can enter up to age 25."},
    {"id": "F03", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "How many adults and children can be covered in a family floater policy?",
     "gold": ["1.1"], "facts": [["4 adults", "four adults"], ["6 dependent children", "six dependent children", "6 children"]],
     "reference": "A family floater policy can include up to 4 adults and 6 dependent children."},
    {"id": "F04", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the minimum duration of hospitalization for a claim to be admissible?",
     "gold": ["3.1.1"], "facts": [["24 consecutive hours", "24 hours"], ["day care"]],
     "reference": "Hospitalization must last at least 24 consecutive hours; the limit does not apply to day care treatment."},
    {"id": "F05", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the waiting period for pre-existing diseases?",
     "gold": ["10.1"], "facts": [["36 months"]],
     "reference": "Pre-existing diseases are excluded until 36 months of continuous coverage (Excl01)."},
    {"id": "F06", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the initial waiting period for illnesses after the policy starts?",
     "gold": ["10.1.3"], "facts": [["30 days"], ["accident"]],
     "reference": "Illness treated within 30 days of the first policy start is excluded, except claims arising from an accident (Excl03)."},
    {"id": "F07", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the grace period for renewing the policy?",
     "gold": ["12.8"], "facts": [["30 days"], ["not available during the grace period", "coverage is not available"]],
     "reference": "The policy can be renewed within a 30-day Grace Period without a break, but there is no coverage during the Grace Period."},
    {"id": "F08", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "How long is the free look period?",
     "gold": ["12.7"], "facts": [["fifteen days", "15 days"]],
     "reference": "New individual policies have a free look period of fifteen days from receipt of the policy document."},
    {"id": "F09", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "Within how many days must the company settle a claim after receiving the last necessary document?",
     "gold": ["12.2"], "facts": [["30 days"], ["45 days"]],
     "reference": "Claims are settled or rejected within 30 days of the last necessary document, or within 45 days when an investigation is needed."},
    {"id": "F10", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the per claim deductible under the Global Health Cover?",
     "gold": ["4.9.1", "4.10.1"], "facts": [["10,000"]],
     "reference": "A per claim deductible of Rs. 10,000 applies to each Global Health Cover claim (except E-Opinion)."},
    {"id": "F11", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the maximum total discount when discounts are combined?",
     "gold": ["8.1"], "facts": [["20%"]],
     "reference": "Employee, online, loyalty and family discounts together cannot exceed 20%."},
    {"id": "F12", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What discount is given to NRI policyholders?",
     "gold": ["8.2"], "facts": [["40%"]],
     "reference": "Insured persons living abroad for the whole policy year get a 40% NRI discount."},
    {"id": "F13", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the moratorium period?",
     "gold": ["12.5"], "facts": [["eight continuous years", "8 continuous years", "eight years", "8 years"]],
     "reference": "After eight continuous years under the policy no look back applies, except for proven fraud and permanent exclusions."},
    {"id": "F14", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "Under which section of the Income Tax Act does the premium qualify for deduction?",
     "gold": ["9"], "facts": [["80d"]],
     "reference": "The premium qualifies for deduction under Section 80D of the Income Tax Act."},
    {"id": "F15", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "How long is a cashless pre-authorization valid?",
     "gold": ["11.5"], "facts": [["15 days"]],
     "reference": "A pre-authorization is valid for 15 days from issue or until the policy expires, whichever is earlier."},
    {"id": "F16", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "How far in advance must a planned hospitalization be notified?",
     "gold": ["11.1"], "facts": [["48 hours"]],
     "reference": "A planned hospitalization must be notified at least 48 hours before admission."},
    {"id": "F17", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "What is the maximum medical underwriting loading?",
     "gold": ["13.4"], "facts": [["100%"], ["150%"]],
     "reference": "Loading cannot exceed 100% per condition and 150% in total per insured person."},
    {"id": "F18", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "How does the Cumulative Bonus work and which plan does it apply to?",
     "gold": ["3.8"], "facts": [["10%"], ["optima suraksha"], ["100%"]],
     "reference": "Cumulative Bonus applies only to the Optima Suraksha plan: 10% of the Base Sum Insured is added for each claim-free year, up to 100%, and reduced at the same rate after a claim."},
    {"id": "F19", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "How early must a portability request be made?",
     "gold": ["12.9"], "facts": [["45 days"], ["60 days"]],
     "reference": "Apply to port at least 45 days, but not earlier than 60 days, before the renewal date."},
    {"id": "F20", "type": "fact", "paraphrase": False, "answerable": True,
     "question": "How much notice is needed for the policyholder to cancel the policy, and what is refunded?",
     "gold": ["12.11"], "facts": [["15 days", "15days"], ["pro rata"]],
     "reference": "The policyholder can cancel with 15 days' written notice and gets a pro-rata refund for the unexpired period, unless a claim was admitted."},

    # ---------------- lists ----------------
    {"id": "L01", "type": "list", "paraphrase": False, "answerable": True,
     "question": "Which expenses are covered under Hospitalization Expenses?",
     "gold": ["3.1"], "facts": [["room rent"], ["intensive care", "icu"], ["surgeon"], ["medicines"], ["prosthetics"]],
     "reference": "Room rent and nursing, ICU/ICCU, surgeon and specialist fees, diagnostics, medicines, IV fluids and blood, operation theatre, and internally implanted prosthetics."},
    {"id": "L02", "type": "list", "paraphrase": False, "answerable": True,
     "question": "Which expenses are excluded from the Organ Donor Expenses cover?",
     "gold": ["3.7"], "facts": [["pre hospitalization"], ["transportation or preservation", "organ transportation"], ["organ harvesting"]],
     "reference": "The donor's pre- and post-hospitalization expenses, organ transportation or preservation, and other expenses consequent to the organ harvesting are excluded."},
    {"id": "L03", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What are the specific exclusions for the Global Health Cover (Emergency Treatments Only)?",
     "gold": ["4.9.2"], "facts": [["planned treatments"], ["oncological", "cancer"], ["orthopedic"], ["sole reason"]],
     "reference": "Planned treatment, pre/post-hospitalization for a paid claim, non-life-threatening treatment that can wait, travel for treatment, orthopedic conditions except fractures and injuries, cancer, and countries under territorial restriction."},
    {"id": "L04", "type": "list", "paraphrase": False, "answerable": True,
     "question": "Which benefits does the Global Health Cover (Emergency & Planned Treatments) extend outside India?",
     "gold": ["4.10"], "facts": [["ayush"], ["pre hospitalization"], ["post hospitalization"], ["organ donor"], ["air ambulance"], ["e opinion"]],
     "reference": "Hospitalization, AYUSH, pre- and post-hospitalization, organ donor, emergency air ambulance, Protect Benefit, Plus Benefit and E-Opinion."},
    {"id": "L05", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What add-on covers are offered with my:Optima Secure?",
     "gold": ["6", "15.1"], "facts": [["critical illness"], ["hospital cash"], ["personal accident"], ["unlimited restore"]],
     "reference": "Critical Illness add-on, Hospital Cash Benefit add-on, Individual Personal Accident Rider and Unlimited Restore."},
    {"id": "L06", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What are the specific exclusions listed in Section 10.2?",
     "gold": ["10.2", "10.2.1"], "facts": [["war"], ["suicide", "self injury"], ["congenital external"], ["stem cell"], ["vaccination"], ["outpatient"], ["hearing aids", "spectacles"]],
     "reference": "War and nuclear risks, deductible amounts, self-injury and suicide, military operations, sleep-apnoea investigation, congenital external diseases, stem cell harvesting, spinal manipulation, circumcision, vaccination, non-medical items, outpatient treatment, hearing aids and spectacles, alopecia, external devices, unreasonable charges and permanent exclusions."},
    {"id": "L07", "type": "list", "paraphrase": False, "answerable": True,
     "question": "Which illnesses count as Major Medical Illnesses for the E-Opinion cover?",
     "gold": ["4.8.1"], "facts": [["51"], ["cancer"], ["kidney failure"], ["stroke"]],
     "reference": "51 conditions, including cancer of specified severity, open chest CABG, kidney failure on dialysis, heart attack, stroke, major organ transplant and paralysis."},
    {"id": "L08", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What additional documents are needed for a claim outside India?",
     "gold": ["11.7.1"], "facts": [["passport"], ["flight tickets", "boarding pass"], ["accommodation invoices"], ["written advice"]],
     "reference": "Passport copy with entry and exit stamps, flight tickets and boarding pass, accommodation invoices, and the overseas doctor's written advice on the need for an accompanying person."},
    {"id": "L09", "type": "list", "paraphrase": False, "answerable": True,
     "question": "Which cities are classified as Tier 1 for premium calculation?",
     "gold": ["14"], "facts": [["delhi"], ["mumbai"], ["thane"], ["surat"], ["ahmedabad"], ["vadodara"]],
     "reference": "Delhi and NCR, Mumbai and Mumbai Suburban, Thane and Navi Mumbai, Surat, Ahmedabad and Vadodara; the rest of India is Tier 2."},
    {"id": "L10", "type": "list", "paraphrase": False, "answerable": True,
     "question": "What can the company do if a pre-existing disease was not disclosed?",
     "gold": ["13"], "facts": [["permanently exclude"], ["4 years"], ["loading"]],
     "reference": "With the policyholder's consent it can permanently exclude the condition, add a waiting period of up to 4 years, or apply underwriting loading."},

    # ---------------- procedures ----------------
    {"id": "P01", "type": "procedure", "paraphrase": False, "answerable": True,
     "question": "What is the procedure for cashless claims outside India?",
     "gold": ["11.3"], "facts": [["release of information"], ["guarantee of payment"], ["network provider"]],
     "reference": "Intimate the claim, take treatment at a Network Provider subject to pre-authorization: the insured signs a Release of Information form, the Company retrieves the hospital documents and then issues a guarantee of payment or rejects the request with reasons."},
    {"id": "P02", "type": "procedure", "paraphrase": False, "answerable": True,
     "question": "How does cashless approval work for Home Health Care?",
     "gold": ["11.4", "3.2"], "facts": [["care plan"], ["authorization letter"]],
     "reference": "After a pre-authorization form is received, the home healthcare provider shares the care plan and cost estimate, and the Company issues an authorization letter with the sanctioned amount or rejects the request."},
    {"id": "P03", "type": "procedure", "paraphrase": False, "answerable": True,
     "question": "How and when must a claim be notified to the insurer?",
     "gold": ["11.1"], "facts": [["24 hours"], ["48 hours"]],
     "reference": "Notify within 24 hours of an emergency hospitalization (or before discharge, if earlier), and at least 48 hours before a planned admission or home health care."},

    # ---------------- comparisons ----------------
    {"id": "C01", "type": "comparison", "paraphrase": False, "answerable": True,
     "question": "How does the maximum single-claim amount for the Optima Secure plan change from Year 1 to Year 3?",
     "gold": ["2.1", "2.2", "2.3"], "facts": [["20 lakh"], ["25 lakh"], ["30 lakh"]],
     "reference": "For Optima Secure the single-claim maximum is Rs. 20 lakh in Year 1, Rs. 25 lakh in Year 2 and Rs. 30 lakh in Year 3 (assuming no claims)."},
    {"id": "C02", "type": "comparison", "paraphrase": False, "answerable": True,
     "question": "What is the difference between the Global Health Cover for emergency treatments only and the one for emergency and planned treatments?",
     "gold": ["4.9", "4.9.1", "4.9.2", "4.10"], "facts": [["planned"], ["pre hospitalization"], ["45 days"]],
     "reference": "The emergency-only cover excludes planned treatment and pre/post-hospitalization for overseas claims and requires treatment to start within 45 days of a trip; the emergency and planned cover also pays planned treatment and overseas pre- and post-hospitalization."},
    {"id": "C03", "type": "comparison", "paraphrase": False, "answerable": True,
     "question": "How do the Cumulative Bonus and the Plus Benefit differ?",
     "gold": ["3.8", "4.4"], "facts": [["10%"], ["50%"], ["claim free"], ["irrespective"]],
     "reference": "Cumulative Bonus (Optima Suraksha only) adds 10% per claim-free year and is reduced after a claim; Plus Benefit adds 50% per renewal irrespective of claims; both are capped at 100% of the Base Sum Insured."},
    {"id": "C04", "type": "comparison", "paraphrase": False, "answerable": True,
     "question": "How do the grace periods differ for monthly and yearly premium instalments?",
     "gold": ["12.12"], "facts": [["15 days"], ["30 days"]],
     "reference": "Monthly instalments have a 15-day grace period; yearly, half-yearly and quarterly instalments have 30 days."},

    # ---------------- conditions and eligibility ----------------
    {"id": "K01", "type": "conditional", "paraphrase": True, "answerable": True,
     "question": "Can a 3-year-old child be insured without a parent on the policy?",
     "gold": ["1"], "facts": [["parent"], ["5 years"]],
     "reference": "No. A child between 91 days and 5 years can be insured only if a parent is also insured; from 5 to 25 years a child can be insured individually."},
    {"id": "K02", "type": "conditional", "paraphrase": True, "answerable": True,
     "question": "Will the air ambulance also fly me back home after treatment?",
     "gold": ["4.1"], "facts": [["return transportation"]],
     "reference": "No. Return transportation to home or elsewhere by air ambulance is not covered."},
    {"id": "K03", "type": "conditional", "paraphrase": False, "answerable": True,
     "question": "When can the Aggregate Deductible be waived or reduced?",
     "gold": ["4.7.1"], "facts": [["50 years"], ["5 continuous"], ["61 years"]],
     "reference": "Once in the policy's lifetime at renewal, if the eldest insured was under 50 when buying, after 5 continuous policy years, and the eldest insured is under 61 when using the option."},
    {"id": "K04", "type": "conditional", "paraphrase": False, "answerable": True,
     "question": "Is Daily Cash for Shared Room paid for time spent in the ICU?",
     "gold": ["4.2"], "facts": [["intensive care", "icu"], ["48"]],
     "reference": "No. The daily cash is not paid for time spent in the ICU; it applies to shared rooms in Network Provider hospitals when hospitalization exceeds 48 hours."},
    {"id": "K05", "type": "conditional", "paraphrase": True, "answerable": True,
     "question": "If I made claims last year, do I still get the extra 50% added when I renew?",
     "gold": ["4.4"], "facts": [["irrespective"], ["50%"]],
     "reference": "Yes. The Plus Benefit of 50% of the Base Sum Insured is added irrespective of the number of claims, up to 100%."},
    {"id": "K06", "type": "conditional", "paraphrase": False, "answerable": True,
     "question": "Does unused Secure Benefit carry forward to the next year?",
     "gold": ["4.5"], "facts": [["not be carried forward", "not carried forward"]],
     "reference": "No. Any unused Secure Benefit is not carried forward; it applies once per policy year after the Base Sum Insured is exhausted."},
    {"id": "K07", "type": "conditional", "paraphrase": True, "answerable": True,
     "question": "Under what conditions will the policy pay for bariatric surgery?",
     "gold": ["10.1.3"], "facts": [["18 years"], ["re:\\b40\\b"], ["re:\\b35\\b"], ["morbidit"]],
     "reference": "Obesity surgery (Excl06) is covered only on a doctor's advice, supported by clinical protocols, for members aged 18 or older with BMI of 40 or more, or 35 or more with a severe co-morbidity after less invasive methods failed."},
    {"id": "K08", "type": "conditional", "paraphrase": True, "answerable": True,
     "question": "Does the 30-day initial waiting period still apply if I have been continuously insured for more than a year?",
     "gold": ["10.1.3"], "facts": [["twelve months", "12 months"]],
     "reference": "No. The 30-day exclusion does not apply if the insured has continuous coverage for more than twelve months."},
    {"id": "K09", "type": "conditional", "paraphrase": True, "answerable": True,
     "question": "Can the insurer refuse to renew my policy because I made claims?",
     "gold": ["12.8"], "facts": [["not be denied"]],
     "reference": "No. Renewal shall not be denied because of claims made in earlier years; it can be refused only for fraud or misrepresentation."},
    {"id": "K10", "type": "conditional", "paraphrase": False, "answerable": True,
     "question": "What happens to the Plus Benefit if the Sum Insured is reduced at renewal?",
     "gold": ["4.4.1"], "facts": [["same proportion"]],
     "reference": "The Plus Benefit is reduced in the same proportion as the Sum Insured."},
    # v2: the Cumulative Bonus notes sit in sections whose labels are sentence fragments
    # ("3.9 In case where the Policy", "3.10 If the Sum Insured has been reduced"): do they get found?
    {"id": "K11", "type": "conditional", "paraphrase": False, "answerable": True,
     "question": "On a family floater policy, how is the Cumulative Bonus added and when is it reduced?",
     "gold": ["3.9"], "facts": [["floater basis"], ["any family member", "any of the insured persons"]],
     "reference": "On a floater policy the Cumulative Bonus is added for the whole family, provided no claim is reported from any family member, and it is reduced after a claim from any insured person."},
    {"id": "K12", "type": "conditional", "paraphrase": False, "answerable": True,
     "question": "What happens to an awarded Cumulative Bonus if a claim from the expiring year is notified after the renewal premium is accepted?",
     "gold": ["3.10"], "facts": [["withdrawn"]],
     "reference": "Any awarded Cumulative Bonus is withdrawn."},

    # ---------------- definitions ----------------
    {"id": "D01", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What is the Protect Benefit?",
     "gold": ["4.3"], "facts": [["non medical expenses"], ["annexure b"]],
     "reference": "Protect Benefit pays the non-medical expenses listed in Annexure B for claims admissible under Section 3."},
    {"id": "D02", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What is the Automatic Restore Benefit?",
     "gold": ["4.6"], "facts": [["base sum insured"], ["once"], ["not be carried forward", "not carried forward"]],
     "reference": "After a claim uses the Base Sum Insured in whole or part, it is restored up to the Base Sum Insured for later claims, once per policy year, and unused amounts are not carried forward."},
    {"id": "D03", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What is domiciliary hospitalization and when is it covered?",
     "gold": ["3.3"], "facts": [["not be removed"], ["non availability of room"]],
     "reference": "Treatment at home, prescribed by a doctor, when the patient could not be moved to a hospital or no hospital room was available."},
    {"id": "D04", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What does the AYUSH Treatment cover include?",
     "gold": ["3.4"], "facts": [["ayurveda"], ["homeopathy"], ["sub limit"]],
     "reference": "In-patient care under Ayurveda, Yoga and Naturopathy, Unani, Siddha and Homeopathy in an AYUSH hospital, up to the sub-limit in the Policy Schedule."},
    {"id": "D05", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What is the E-Opinion for Critical Illness cover?",
     "gold": ["4.8"], "facts": [["once"], ["network provider"], ["sum insured"]],
     "reference": "An electronic second opinion on a listed major illness through the Network Provider, once per policy year per insured person, with no impact on the Sum Insured."},
    {"id": "D06", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What is the Preventive Health Check-up benefit?",
     "gold": ["5"], "facts": [["renewal"], ["not carry forward"]],
     "reference": "On each continuous renewal the cost of a health check-up is reimbursed up to the specified limit; it does not carry forward if unclaimed."},
    {"id": "D07", "type": "definition", "paraphrase": False, "answerable": True,
     "question": "What is Home Health Care cover?",
     "gold": ["3.2"], "facts": [["pre authorized"], ["not available on reimbursement"]],
     "reference": "Treatment at home that would otherwise need in-patient care, if prescribed and pre-authorized, with daily signed records; it is not available on reimbursement basis."},

    # ---------------- answers that span several sections ----------------
    {"id": "M01", "type": "multi_section", "paraphrase": True, "answerable": True,
     "question": "In what order is the Sum Insured used up when I make claims?",
     "gold": ["13.2"], "facts": [["base sum insured"], ["plus benefit", "cumulative bonus"], ["secure benefit"], ["automatic restore"]],
     "reference": "Base Sum Insured first, then Cumulative Bonus or Plus Benefit, then Secure Benefit, then Automatic Restore."},
    {"id": "M02", "type": "multi_section", "paraphrase": False, "answerable": True,
     "question": "Which covers are not subject to the Aggregate Deductible?",
     "gold": ["4.7", "4.9.1", "4.10.1"], "facts": [["e opinion"], ["preventive health"], ["global health cover"], ["overseas travel secure"]],
     "reference": "E-Opinion for Critical Illness, Preventive Health Check-up, both Global Health Covers and Overseas Travel Secure."},
    {"id": "M03", "type": "multi_section", "paraphrase": True, "answerable": True,
     "question": "Does this policy cover treatment taken outside India?",
     "gold": ["13.3", "4.9", "4.10"], "facts": [["global health cover"], ["india"]],
     "reference": "The policy covers India only, except through the optional Global Health Covers (Sections 4.9 and 4.10), Overseas Travel Secure and E-Opinion."},
    {"id": "M04", "type": "multi_section", "paraphrase": False, "answerable": True,
     "question": "What limits and conditions apply to claims under the Global Health Cover?",
     "gold": ["4.9.1", "4.10.1"], "facts": [["base sum insured and plus benefit"], ["10,000"], ["reimbursement"], ["exchange"]],
     "reference": "Liability is capped at the Base Sum Insured plus Plus Benefit, a Rs. 10,000 deductible applies per claim, claims are normally reimbursed (cashless case by case) at the RBI exchange rate, in registered hospitals."},

    # ---------------- everyday wording (no heading words; tests retrieval beyond title matching) ----------------
    {"id": "X01", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "How long do I have to wait before kidney stone treatment is covered?",
     "gold": ["10.1", "10.1.1"], "facts": [["24 months"]],
     "reference": "Kidney stones are a listed condition under Excl02, so treatment is covered after 24 months of continuous coverage, unless caused by an accident."},
    {"id": "X02", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Is cataract surgery covered from day one?",
     "gold": ["10.1", "10.1.1"], "facts": [["24 months"]],
     "reference": "No. Cataract is a listed condition with a 24-month waiting period (Excl02)."},
    {"id": "X03", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Will the policy pay for my wife's delivery?",
     "gold": ["10.1.5"], "facts": [["excl18", "maternity"], ["childbirth"]],
     "reference": "No. Childbirth expenses, including caesarean sections, are excluded under Excl18 (Maternity), except ectopic pregnancy."},
    {"id": "X04", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Are IVF treatments reimbursed?",
     "gold": ["10.1.5"], "facts": [["excl17", "infertility", "sterility"]],
     "reference": "No. Assisted reproduction such as IVF is excluded under Excl17 (Sterility and Infertility)."},
    {"id": "X05", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "I got hurt while scuba diving as a professional diver. Will I be covered?",
     "gold": ["10.1.4"], "facts": [["excl09", "hazardous", "adventure"]],
     "reference": "No. Treatment needed because of professional participation in hazardous or adventure sports, including scuba diving, is excluded (Excl09)."},
    {"id": "X06", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Can I claim the cost of new spectacles?",
     "gold": ["10.2.1"], "facts": [["spectacles"]],
     "reference": "No. Provision or fitting of hearing aids, spectacles or contact lenses is excluded."},
    {"id": "X07", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "For how many days before admission is pre hospitalisation covered?",
     "gold": ["3.5"], "facts": [["60 days"]],
     "reference": "Up to 60 days before an admissible hospitalization."},
    {"id": "X08", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "How many days of post hospitalisation expenses can I claim after discharge?",
     "gold": ["3.6"], "facts": [["180 days"]],
     "reference": "Up to 180 days from the date of discharge."},
    {"id": "X09", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Do I pay less if I buy the policy on the website?",
     "gold": ["8"], "facts": [["5%"]],
     "reference": "Yes. Buying online gives a 5% discount, which continues on renewals without an intermediary."},
    {"id": "X10", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "My son is turning 25. What happens to his cover?",
     "gold": ["1"], "facts": [["migrated", "new policy"], ["continuity"]],
     "reference": "At 25 he becomes ineligible on the next renewal and is migrated to a new policy with continuity benefits."},
    {"id": "X11", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Does the policy pay for an ambulance to take me to the hospital?",
     "gold": ["3.1.1"], "facts": [["road ambulance", "ambulance"], ["nearest hospital"]],
     "reference": "Yes. Road ambulance costs are covered for transfer to the nearest hospital for emergency care, between hospitals, or from hospital to home in the same city."},
    {"id": "X12", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Is dental treatment covered?",
     "gold": ["3.1.1"], "facts": [["dental"], ["disease or injury"]],
     "reference": "In-patient dental treatment is covered when necessitated by disease or injury."},
    {"id": "X13", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "How soon will I get paid after submitting my claim papers?",
     "gold": ["12.2"], "facts": [["30 days"]],
     "reference": "Within 30 days of the last necessary document, or 45 days if investigated, with interest at 2% above the bank rate for delays."},
    {"id": "X14", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "If I move my policy to another insurer, do I keep the waiting period I have already served?",
     "gold": ["12.9"], "facts": [["continuity"]],
     "reference": "Yes. With continuous coverage, the accrued continuity benefits in waiting periods carry over as per IRDAI portability guidelines."},
    {"id": "X15", "type": "paraphrase", "paraphrase": True, "answerable": True,
     "question": "Can I pay my premium every month?",
     "gold": ["12.12"], "facts": [["monthly"], ["15 days"]],
     "reference": "Yes. Monthly instalments are allowed, with a 15-day grace period during which there is no cover."},

    # ---------------- not answerable from the policy ----------------
    {"id": "N01", "type": "out_of_scope", "paraphrase": False, "answerable": False,
     "question": "Does this policy cover damage to my car in an accident?",
     "gold": [], "facts": [], "absent_terms": ["vehicle damage", "car damage", "own damage"],
     "reference": "Not specified in the provided policy text."},
    {"id": "N02", "type": "out_of_scope", "paraphrase": False, "answerable": False,
     "question": "How much does the policy pay for lost baggage during travel?",
     "gold": [], "facts": [], "absent_terms": ["baggage", "luggage"],
     "reference": "Not specified in the provided policy text."},
    {"id": "N03", "type": "out_of_scope", "paraphrase": False, "answerable": False,
     "question": "What is the interest rate on a personal loan from HDFC?",
     "gold": [], "facts": [], "absent_terms": ["loan"],
     "reference": "Not specified in the provided policy text."},
    {"id": "N04", "type": "out_of_scope", "paraphrase": False, "answerable": False,
     "question": "What does the Optima Restore plan cover?",
     "gold": [], "facts": [], "absent_terms": ["optima restore"],
     "reference": "Not specified in the provided policy text."},
    {"id": "N05", "type": "out_of_scope", "paraphrase": False, "answerable": False,
     "question": "What is the annual premium for a 40-year-old living in a Tier 2 city?",
     "gold": [], "facts": [], "absent_terms": ["premium for a 40", "40-year-old"],
     "reference": "Not specified in the provided policy text (the premium table, Annexure C, is not included)."},
    {"id": "N06", "type": "out_of_scope", "paraphrase": False, "answerable": False,
     "question": "Does the policy cover home contents against fire?",
     "gold": [], "facts": [], "absent_terms": ["fire", "home contents", "burglary"],
     "reference": "Not specified in the provided policy text."},

    # ---------------- false premises ----------------
    {"id": "FP01", "type": "false_premise", "paraphrase": False, "answerable": False,
     "question": "What is the co-payment percentage for senior citizens?",
     "gold": [], "facts": [], "absent_terms": ["co-payment for senior", "senior citizen co-pay"],
     "reference": "Not specified in the provided policy text: no senior-citizen co-payment is stated."},
    {"id": "FP02", "type": "false_premise", "paraphrase": False, "answerable": False,
     "question": "What is the room rent limit of 1% of the Sum Insured per day?",
     "gold": [], "facts": [], "absent_terms": ["1% of the sum insured", "room rent limit"],
     "reference": "Not specified in the provided policy text: no room rent cap of 1% of the Sum Insured is stated."},
    {"id": "FP03", "type": "false_premise", "paraphrase": True, "answerable": True,
     "question": "How much is the maternity benefit after the 9-month waiting period?",
     "gold": ["10.1.5"], "facts": [["excl18", "excluded", "not covered"]],
     "reference": "There is no maternity benefit: childbirth expenses are excluded under Excl18, and no 9-month waiting period is stated."},
]

# ---------------------------------------------------------------- fact matching
def norm_text(s: str) -> str:
    """Lower-case; drop currency marks; hyphens and dashes to spaces; British -isation/-ised to
    -ization/-ized; collapse whitespace. Applied to the fact phrases and to the text searched."""
    s = s.lower().replace("₹", " ")
    s = re.sub(r"\brs\.?\s*", " ", s)
    s = re.sub(r"[-‐‑‒–—/]", " ", s)
    s = re.sub(r"isation", "ization", s)
    s = re.sub(r"ised\b", "ized", s)
    return re.sub(r"\s+", " ", s).strip()

def fact_found(alternatives, text_norm: str) -> bool:
    for alt in alternatives:
        if alt.startswith("re:"):
            if re.search(alt[3:], text_norm):
                return True
        elif norm_text(alt) in text_norm:
            return True
    return False

def fact_recall(facts, text: str):
    """Share of key facts present in a text (a context or an answer); None if there are no facts."""
    if not facts:
        return None
    t = norm_text(text or "")
    return sum(fact_found(f, t) for f in facts) / len(facts)

def check_testset(testset=None):
    """Checks the test set against the policy text loaded in Section 1: every gold section exists and
    holds its facts, and no out-of-scope question's key terms occur in the policy. Run it again if
    you change the policy file or the parser: a failure here means the test set needs updating."""
    testset = testset or TESTSET
    by_id = {s["section_id"]: s for s in sections}
    doc = norm_text(" ".join(s["text"] for s in sections))
    problems = []
    for t in testset:
        missing = [g for g in t["gold"] if g not in by_id]
        if missing:
            problems.append((t["id"], "gold section not found", missing))
        gold_text = norm_text(" ".join(by_id[g]["text"] for g in t["gold"] if g in by_id))
        for f in t["facts"]:
            if not fact_found(f, gold_text):
                problems.append((t["id"], "fact not in its gold sections", f))
        for term in t.get("absent_terms", []):
            if re.search(r"\b" + re.escape(norm_text(term)) + r"\b", doc):
                problems.append((t["id"], "out-of-scope term occurs in the policy", term))
    kinds = pd.Series([t["type"] for t in testset]).value_counts().to_dict()
    print(f"Test set: {len(testset)} questions ({sum(t['answerable'] for t in testset)} answerable, "
          f"{sum(t['paraphrase'] for t in testset)} in everyday wording), "
          f"{sum(len(t['facts']) for t in testset)} key facts | types: {kinds}")
    print("Check against the loaded policy:", "all facts found, all out-of-scope terms absent" if not problems
          else f"{len(problems)} problem(s)")
    for p in problems:
        print("  ", p)
    return problems

_ = check_testset()

## 10.4 Evaluation workflow: runner and metrics

`run_experiment(name, overrides, mode)` answers every test question with the given `CONFIG`
overrides, scores each one, saves the run to `eval_runs/` and restores all settings afterwards.
`mode="retrieval"` stops before the LLM (about 1 s per question); `mode="full"` generates answers.

| Group | Metric | What it measures | Mode |
|---|---|---|---|
| Retrieval | `ctx_fact_recall` | Share of the key facts present in the context the model sees: **the main retrieval metric** | both |
| | `ctx_section_recall` / `cand_section_recall` | Share of the gold sections in the context / among the search candidates | both |
| | `gold_rr` | 1 / rank of the first gold-section chunk after reranking (its mean is the MRR) | both |
| Routing | `answered` / `refused` | Gate decision on answerable / not-answerable questions | both |
| Correctness | `answer_fact_recall` | Share of the key facts present in the answer: **the main answer metric** | full |
| | `false_refusal` / `refusal_ok` | Answerable question refused / not-answerable question refused or answered "not specified" | full |
| Faithfulness | `unsupported_figures` | Amounts, percentages and durations in the answer that are in neither the context nor the question | full |
| | `unsupported_citations` | Section numbers the answer cites that do not occur in the context | full |
| | `faithful` | 1 if neither of the two above occurs | full |
| | `grounding` | Share of the answer's 5-word sequences found in the context | full |
| Reliability | `error` / `oom_retry` | A pipeline node failed / the answer came from the shortened out-of-memory retry | both / full |
| Cost | `context_tokens` | Context size, section labels included | both |

`scorecard(run)` shows one run grouped as above. `compare_runs(baseline, experiment)` pairs the runs
question by question and reports the mean difference with a 95% bootstrap interval, saying *better*
or *worse* only when the interval excludes zero; `regressions()` lists the questions that got worse.
With 85 answerable questions, differences under about 0.03 are usually noise. Retrieval-mode numbers
are deterministic; full-mode answers use greedy decoding, so they repeat on the same GPU and library
versions.

**Keeping a baseline across sessions:** Colab deletes `eval_runs/` when the runtime is recycled.
`export_runs()` downloads the runs as a zip; in a later session `upload_runs()` puts them back and
`load_run_file("eval_runs/baseline__full.json")` loads one. Runs record the test-set version, and
comparisons warn when two runs used different versions.

**Limits:** fact and figure matching are string-based. They can miss a correct answer worded
differently ("two years" for "24 months") and cannot judge whether a fact is used correctly, so read
the questions listed as regressions before adopting a change.

In [ ]:
# ================================================================
# 10.4) Experiment runner and metrics
# ================================================================
# run_experiment() answers every test question with CONFIG overrides applied, scores each answer,
# saves the run to eval_runs/, and restores everything afterwards.
#   mode="retrieval": retrieve + rerank + context + gate only (no LLM): about 1 s per question on a T4
#   mode="full":      the whole pipeline (app.invoke), answers included
import time, datetime, hashlib, gc, contextlib

EVAL_DIR = Path("eval_runs")
EVAL_DIR.mkdir(exist_ok=True)
TESTSET_VERSION = hashlib.sha1(json.dumps(TESTSET, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:10]
NOTEBOOK_VERSION = "v2.6"
_MISSING = object()
_HEDGE = re.compile(r"not specified|does not (specify|mention|cover|state|provide|include)|"
                    r"no (information|mention|details)|not (mentioned|covered|stated|provided|included)", re.I)

# ---------------------------------------------------------------- resources an experiment may swap
def _free_gpu_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    free, _ = torch.cuda.mem_get_info()
    return free / 1e9

@contextlib.contextmanager
def _swap_globals(**new):
    g = globals()
    old = {k: g[k] for k in new}
    g.update(new)
    try:
        yield
    finally:
        g.update(old)

@contextlib.contextmanager
def _embedder(name: str):
    """Another embedding model with its own FAISS index over the same chunks (Section 4 text)."""
    dev = "cuda" if _free_gpu_gb() > 2.0 else "cpu"
    model = SentenceTransformer(name, device=dev)
    emb = model.encode(embed_texts, batch_size=32, convert_to_numpy=True,
                       show_progress_bar=False).astype("float32")
    faiss.normalize_L2(emb)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    print(f"  [setup] embedder {name} on {dev}: {idx.ntotal} chunks indexed")
    try:
        with _swap_globals(embed_model=model, index=idx):
            yield
    finally:
        del model, idx
        gc.collect()
        free_gpu()

@contextlib.contextmanager
def _reranker(name: str):
    """Another cross-encoder. Runs on the GPU in fp16 when there is room, otherwise on the CPU."""
    dev = "cuda" if _free_gpu_gb() > 3.0 else "cpu"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(
        name, torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
    print(f"  [setup] reranker {name} on {dev}" + (" (slow: GPU is full)" if dev == "cpu" else ""))
    try:
        with _swap_globals(tok_rerank=tok, reranker=model, device=dev):
            yield
    finally:
        del model, tok
        gc.collect()
        free_gpu()

@contextlib.contextmanager
def experiment_setup(overrides: dict):
    """Apply CONFIG overrides plus what they need (a BM25 rebuild, another embedder or reranker),
    and undo all of it afterwards, even if the run fails."""
    saved = {k: CONFIG.get(k, _MISSING) for k in overrides}
    stack = contextlib.ExitStack()
    try:
        CONFIG.update(overrides)
        if "bm25_stemming" in overrides:
            build_bm25()
        if "embed_model" in overrides and overrides["embed_model"] != saved["embed_model"]:
            stack.enter_context(_embedder(overrides["embed_model"]))
        if "reranker_model" in overrides and overrides["reranker_model"] != saved["reranker_model"]:
            stack.enter_context(_reranker(overrides["reranker_model"]))
        yield
    finally:
        stack.close()
        for k, v in saved.items():
            if v is _MISSING:
                CONFIG.pop(k, None)
            else:
                CONFIG[k] = v
        if "bm25_stemming" in overrides:
            build_bm25()

# ---------------------------------------------------------------- faithfulness checks
# Figures (amounts, percentages, durations) and section citations in a text, for faithfulness checks.
_WORD_NUM = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
             "nine": 9, "ten": 10, "twelve": 12, "fifteen": 15, "twenty": 20, "thirty": 30,
             "forty five": 45, "sixty": 60, "ninety": 90}
_UNIT = {"day": "days", "month": "months", "year": "years", "hour": "hours"}

def figures(text: str) -> set:
    """Amounts in rupees (lakh/crore and digit forms unified), percentages and durations,
    e.g. {("inr", 1500000), ("pct", 50.0), ("days", 60)}. Word numbers ("fifteen days") count."""
    t = (text or "").lower().replace("₹", " ₹ ").replace("-", " ")
    for w, n in sorted(_WORD_NUM.items(), key=lambda x: -len(x[0])):
        t = re.sub(rf"\b{w}\b", str(n), t)
    out = set()
    for m in re.finditer(r"(?:₹|\brs\.?|\binr)?\s*(\d+(?:\.\d+)?)\s*(lakhs?|lacs?|crores?|cr)\b", t):
        mult = 1e5 if m.group(2).startswith("la") else 1e7
        out.add(("inr", int(round(float(m.group(1)) * mult))))
    for m in re.finditer(r"(?:₹|\brs\.?|\binr)\s*(\d[\d,]*(?:\.\d+)?)(?!\d|[.,]\d)(?!\s*(?:lakh|lac|crore|cr)\b)", t):
        out.add(("inr", int(round(float(m.group(1).replace(",", ""))))))
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s*%", t):
        out.add(("pct", float(m.group(1))))
    for m in re.finditer(r"(\d+)\s*(?:consecutive\s+|continuous\s+|calendar\s+)?(day|month|year|hour)s?\b", t):
        out.add((_UNIT[m.group(2)], int(m.group(1))))
    return out

def unsupported_figures(answer: str, context: str, question: str = "") -> set:
    """Figures in the answer that appear neither in the context nor in the question."""
    return figures(answer) - figures(context) - figures(question)

def cited_sections(text: str) -> set:
    """Section numbers cited as 'Section 4.4.1' / 'Section 11.2.a' / 'Sections 3.5 and 3.6'."""
    ids = set()
    for m in re.finditer(r"sections?\s+((?:\d+(?:\.\d+)*(?:\.[a-z])?(?:\s*(?:,|and|&)\s*)?)+)", (text or "").lower()):
        for sid in re.findall(r"\d+(?:\.\d+)*", m.group(1)):
            ids.add(sid.rstrip("."))
    return ids

def unsupported_citations(answer: str, context: str) -> set:
    """Cited section numbers that do not occur anywhere in the context (labels or text)."""
    ctx_ids = set(re.findall(r"\d+(?:\.\d+)+|\b\d+\b", context or ""))
    return {s for s in cited_sections(answer) if s not in ctx_ids}

# ---------------------------------------------------------------- metrics
def _finite(x):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return x if np.isfinite(x) else None

def score_record(rec: dict, item: dict) -> dict:
    """Per-question metrics. Retrieval metrics need no LLM; answer metrics need mode="full"."""
    m = {"context_tokens": rec["context_tokens"], "error": float(bool(rec.get("error")))}
    gold = set(item["gold"])
    cand_secs = {final_chunks[i]["section_id"] for i in rec["candidates"]}
    ctx_secs = {final_chunks[i]["section_id"] for i in rec["context_chunks"]}
    if item["answerable"]:
        m["cand_section_recall"] = len(cand_secs & gold) / len(gold)    # did search find the sections?
        m["ctx_section_recall"] = len(ctx_secs & gold) / len(gold)      # did they reach the model?
        m["ctx_fact_recall"] = fact_recall(item["facts"], rec["context"])   # did the key facts reach it?
        m["gold_rr"] = 0.0                                              # 1 / rank of the first gold chunk
        for rank, (i, _) in enumerate(rec["ranked"], start=1):
            if final_chunks[i]["section_id"] in gold:
                m["gold_rr"] = 1.0 / rank
                break
        m["answered"] = float(rec["gate_route"] == "answer")
    else:
        m["refused"] = float(rec["gate_route"] == "no_answer")
    ans = rec.get("answer")
    if ans is not None:
        m["oom_retry"] = float(bool(rec.get("oom_retry")))
        if not is_no_answer(ans):              # faithfulness: does the answer stay within its context?
            m["unsupported_figures"] = len(unsupported_figures(ans, rec["context"], item["question"]))
            m["unsupported_citations"] = len(unsupported_citations(ans, rec["context"]))
            m["faithful"] = float(m["unsupported_figures"] == 0 and m["unsupported_citations"] == 0)
        if item["answerable"]:
            m["false_refusal"] = float(is_no_answer(ans))
            m["answer_fact_recall"] = fact_recall(item["facts"], ans)
            m["grounding"] = None if is_no_answer(ans) else grounding(ans, rec["context"])
            m["answer_words"] = len(ans.split())
        else:   # a refusal, or an answer saying the policy does not cover or specify it
            m["refusal_ok"] = float(is_no_answer(ans) or bool(_HEDGE.search(ans)))
    return m

# ---------------------------------------------------------------- running
def _config_snapshot():
    return {k: v for k, v in CONFIG.items() if isinstance(v, (str, int, float, bool, type(None)))}

def run_experiment(name: str, overrides: dict = None, mode: str = "retrieval", testset=None,
                   save: bool = True, verbose: bool = True) -> dict:
    assert mode in ("retrieval", "full"), mode
    testset = testset or TESTSET
    overrides = dict(overrides or {})
    if mode == "full" and overrides.get("context_token_budget", CONFIG["context_token_budget"]) > 1500 \
            and not CONFIG["llm_load_in_4bit"]:
        print("  WARNING: a context over 1,500 tokens with the fp16 model will hit the out-of-memory "
              "retry on a T4; load the model in 4-bit (CONFIG['llm_load_in_4bit'] = True) first.")
    calls = []
    def recording(fn):
        def wrapped(query, docs, *a, **kw):
            order, scores = fn(query, docs, *a, **kw)
            calls.append([float(s) for s in scores])
            return order, scores
        return wrapped

    t_run = time.perf_counter()
    records = []
    with experiment_setup(overrides), _swap_globals(rerank_all=recording(globals()["rerank_all"])):
        config = _config_snapshot()
        for n, item in enumerate(testset, start=1):
            calls.clear()
            t0 = time.perf_counter()
            if mode == "full":
                result = app.invoke({"query": item["question"]})
            else:
                result = {"query": item["question"]}
                result.update(retrieve_node(result))
                result.update(rerank_node(result))
            cands = result.get("retrieved", [])
            scores = calls[0] if calls else []
            groups = [[int(i) for i in g] for g in result.get("groups", [])]
            ctx_ids = [i for g in groups for i in g]
            rec = {
                "id": item["id"], "type": item["type"], "paraphrase": item["paraphrase"],
                "answerable": item["answerable"], "question": item["question"],
                "candidates": [int(c["idx"]) for c in cands],
                "ranked": sorted(([int(c["idx"]), round(s, 4)] for c, s in zip(cands, scores)),
                                 key=lambda x: -x[1]),
                "groups": groups, "context_chunks": ctx_ids, "context": format_groups(groups),
                "context_tokens": sum(block_tokens(final_chunks[i]) for i in ctx_ids),
                "top_score": _finite(result.get("top_score")),
                "gate_route": gate(result),
                "answer": result.get("answer") if mode == "full" else None,
                "error": result.get("error"),             # a node failed (it then returns empty results)
                "oom_retry": bool(result.get("oom_retry")),   # answered from a shortened context
                "seconds": round(time.perf_counter() - t0, 2),
            }
            rec["metrics"] = score_record(rec, item)
            records.append(rec)
            if verbose and (n % 20 == 0 or n == len(testset)):
                print(f"  {name} [{mode}] {n}/{len(testset)} questions, {time.perf_counter() - t_run:.0f}s")
    n_err = sum(bool(r["error"]) for r in records)
    if n_err:
        print(f"  WARNING: {n_err} question(s) hit a pipeline error; see run_table(run).query('error > 0')")
    run = {"name": name, "mode": mode, "overrides": overrides, "config": config,
           "testset_version": TESTSET_VERSION, "notebook_version": NOTEBOOK_VERSION,
           "started": datetime.datetime.now().isoformat(timespec="seconds"),
           "seconds": round(time.perf_counter() - t_run, 1), "records": records}
    if save:
        save_run(run)
    return run

def save_run(run: dict) -> Path:
    path = EVAL_DIR / f"{run['name']}__{run['mode']}.json"
    path.write_text(json.dumps(run, ensure_ascii=False))
    return path

def load_run(name: str, mode: str = "retrieval") -> dict:
    return json.loads((EVAL_DIR / f"{name}__{mode}.json").read_text())

def load_run_file(path) -> dict:
    """Load a run saved in an earlier session (e.g. a baseline you downloaded with export_runs())."""
    run = json.loads(Path(path).read_text())
    if run.get("testset_version") != TESTSET_VERSION:
        print(f"  NOTE: {Path(path).name} was made with a different test set "
              f"({run.get('testset_version')} vs {TESTSET_VERSION}); only shared questions are compared.")
    return run

# ---------------------------------------------------------------- reading results
HEADLINE = ["ctx_fact_recall", "ctx_section_recall", "cand_section_recall", "gold_rr", "answered",
            "refused", "answer_fact_recall", "false_refusal", "refusal_ok", "faithful",
            "unsupported_figures", "unsupported_citations", "grounding", "error", "oom_retry",
            "context_tokens"]
LOWER_IS_BETTER = {"false_refusal", "context_tokens", "answer_words", "unsupported_figures",
                   "unsupported_citations", "error", "oom_retry"}
SCORECARD = {
    "Retrieval (what reaches the model)": ["ctx_fact_recall", "ctx_section_recall", "cand_section_recall", "gold_rr"],
    "Routing": ["answered", "refused"],
    "Answer correctness": ["answer_fact_recall", "false_refusal", "refusal_ok"],
    "Faithfulness": ["faithful", "unsupported_figures", "unsupported_citations", "grounding"],
    "Reliability and cost": ["error", "oom_retry", "context_tokens"],
}

def scorecard(run: dict) -> pd.DataFrame:
    """One table per run: each metric's mean and how many questions it covers."""
    t = run_table(run)
    rows = []
    for group, cols in SCORECARD.items():
        for c in cols:
            if c in t and t[c].notna().any():
                rows.append({"group": group, "metric": c, "mean": round(t[c].mean(), 3),
                             "questions": int(t[c].notna().sum()),
                             "better": "lower" if c in LOWER_IS_BETTER else "higher"})
    return pd.DataFrame(rows)

def run_table(run: dict) -> pd.DataFrame:
    rows = [{"id": r["id"], "type": r["type"], "paraphrase": r["paraphrase"],
             "answerable": r["answerable"], **r["metrics"]} for r in run["records"]]
    return pd.DataFrame(rows)

def summarize(run: dict, by: str = None) -> pd.DataFrame:
    t = run_table(run)
    cols = [c for c in HEADLINE if c in t]
    if by is None:
        return t[cols].mean().round(3).to_frame(run["name"]).T
    return t.groupby(by)[cols].mean().round(3)

def experiment_table(runs) -> pd.DataFrame:
    rows = []
    for run in runs:
        s = summarize(run).iloc[0].to_dict()
        rows.append({"experiment": run["name"], "mode": run["mode"], **s, "minutes": round(run["seconds"] / 60, 1)})
    return pd.DataFrame(rows)

def compare_runs(base: dict, new: dict, metrics=None, n_boot: int = 2000, seed: int = 0) -> pd.DataFrame:
    """Paired comparison on the questions both runs answered. ci = 95% bootstrap interval of the mean
    per-question difference; 'better' / 'worse' only when the interval excludes zero."""
    if base.get("testset_version") != new.get("testset_version"):
        print(f"  NOTE: different test set versions ({base.get('testset_version')} vs "
              f"{new.get('testset_version')}); comparing the questions both runs share.")
    a, b = run_table(base).set_index("id"), run_table(new).set_index("id")
    rng = np.random.default_rng(seed)
    rows = []
    for m in metrics or HEADLINE:
        if m not in a or m not in b:
            continue
        pair = pd.concat([a[m], b[m]], axis=1, keys=["a", "b"], join="inner").dropna()
        if pair.empty:
            continue
        d = (pair["b"] - pair["a"]).to_numpy(float)
        boots = rng.choice(d, size=(n_boot, len(d)), replace=True).mean(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        sign = -1 if m in LOWER_IS_BETTER else 1
        gain_lo, gain_hi = sorted((sign * lo, sign * hi))       # interval of the improvement
        verdict = "better" if gain_lo > 0 else "worse" if gain_hi < 0 else "no clear change"
        col_b = new["name"] if new["name"] != base["name"] else new["name"] + f" ({new['mode']})"
        rows.append({"metric": m, "n": len(d), base["name"]: round(pair["a"].mean(), 3),
                     col_b: round(pair["b"].mean(), 3), "diff": round(d.mean(), 3),
                     "ci_low": round(lo, 3), "ci_high": round(hi, 3),
                     "questions_up": int((d > 1e-9).sum()), "questions_down": int((d < -1e-9).sum()),
                     "verdict": verdict})
    return pd.DataFrame(rows)

def regressions(base: dict, new: dict, metric: str = "ctx_fact_recall") -> pd.DataFrame:
    """Questions where the experiment scores lower than the baseline on one metric."""
    a, b = run_table(base).set_index("id"), run_table(new).set_index("id")
    if metric not in a or metric not in b:
        return pd.DataFrame()
    d = (b[metric] - a[metric]).dropna()
    d = d[d < -1e-9] if metric not in LOWER_IS_BETTER else d[d > 1e-9]
    q = {r["id"]: r["question"] for r in base["records"]}
    return pd.DataFrame({"question": [q[i][:70] for i in d.index], "baseline": a.loc[d.index, metric],
                         new["name"]: b.loc[d.index, metric]}, index=d.index)

def routing_sweep(run: dict, thresholds=(-8, -6, -5, -4, -3, -2, -1, 0)) -> pd.DataFrame:
    """Gate decisions at other thresholds, recomputed from the saved best scores (no rerun).
    Use it to re-tune rerank_score_threshold after changing the reranker."""
    rows = []
    for thr in thresholds:
        ans = [r for r in run["records"] if r["answerable"]]
        neg = [r for r in run["records"] if not r["answerable"]]
        route = lambda r: bool(r["groups"]) and r["top_score"] is not None and r["top_score"] >= thr
        rows.append({"threshold": thr, "answered_when_answerable": round(np.mean([route(r) for r in ans]), 3),
                     "refused_when_not_answerable": round(np.mean([not route(r) for r in neg]), 3)})
    return pd.DataFrame(rows)

def export_runs():
    """Zip eval_runs/ and download it in Colab (the runtime's files disappear when it is recycled)."""
    import shutil
    path = shutil.make_archive("eval_runs", "zip", EVAL_DIR)
    if IN_COLAB:
        from google.colab import files
        files.download(path)
    return path

def upload_runs():
    """Colab: upload a zip made by export_runs() (or single run files) back into eval_runs/."""
    if not IN_COLAB:
        print("Copy the files into", EVAL_DIR.resolve())
        return
    import zipfile, io
    from google.colab import files
    for fname, data in files.upload().items():
        if fname.endswith(".zip"):
            zipfile.ZipFile(io.BytesIO(data)).extractall(EVAL_DIR)
        else:
            (EVAL_DIR / fname).write_bytes(data)
    print("eval_runs/ now holds:", sorted(p.name for p in EVAL_DIR.glob("*.json")))

## 10.5 Experiments

Each experiment changes one thing and runs the same questions. Retrieval mode covers everything that
changes what reaches the model; full mode is needed for prompt or model changes and to confirm a
winner on the answers themselves.

- **Ablations** (`v22_like`, `no_title_pinning`, `no_neighbour_expansion`, `no_seed_floor`) show
  whether each v2.3 change helps with the real models.
- **Fixes** (`pin_rules_v25`, `bm25_stemming`, `continuations`, and `all_fixes` for the three
  together) target defects measured offline.
- **BM25** (`dense_only`, `dense_only_k64`, `bm25_only`): remove BM25 only if `dense_only_k64` is not
  worse than the baseline on `ctx_fact_recall` and routing, with no regression on questions that cite
  codes or section numbers.
- **`budget_2600`** shows how much more evidence a larger context lets in; answering with it needs
  the 4-bit model on a T4.
- **Model swaps** (`reranker_v2_m3`, `embed_bge_base`) are off by default: they download extra models
  and need GPU room, so run them with `CONFIG["llm_load_in_4bit"] = True`. A different reranker scores
  on a different scale, so re-tune the gate with `routing_sweep()` first.

In [ ]:
# ================================================================
# 10.5) Experiments — the same questions, one change at a time
# ================================================================
EXPERIMENTS = {
    # name: (CONFIG overrides, what it tests)
    "baseline":               ({}, "current defaults (v2.4 behaviour)"),
    "v22_like":               ({"title_pinning": False, "neighbour_expansion": False,
                                "seed_score_floor": -1e9, "prompt_version": "original"},
                               "the three v2.3 context changes and the scope rules switched off"),
    "no_title_pinning":       ({"title_pinning": False}, "ablation: title pinning off"),
    "no_neighbour_expansion": ({"neighbour_expansion": False}, "ablation: neighbour expansion off"),
    "no_seed_floor":          ({"seed_score_floor": -1e9}, "ablation: low-scoring chunks may fill the context"),
    "pin_rules_v25":          ({"title_pinning_rules": "v2.5"}, "fix: precise title pinning"),
    "bm25_stemming":          ({"bm25_stemming": True}, "fix: stemmed BM25 (notified ~ notification)"),
    "continuations":          ({"continuation_sections": True},
                               "fix: notes numbered as separate sections travel with their section"),
    "all_fixes":              ({"title_pinning_rules": "v2.5", "bm25_stemming": True,
                                "continuation_sections": True}, "the three fixes together"),
    "dense_only":             ({"use_bm25": False}, "BM25 off: dense search only, top 32"),
    "dense_only_k64":         ({"use_bm25": False, "k_dense": 64},
                               "BM25 off, dense top 64: the reranker keeps a pool of the same size"),
    "bm25_only":              ({"k_dense": 0}, "reference: BM25 only (shows what dense search adds)"),
    "budget_2600":            ({"context_token_budget": 2600},
                               "how much a bigger context would add (in full mode it needs the 4-bit model)"),
    "reranker_v2_m3":         ({"reranker_model": "BAAI/bge-reranker-v2-m3"},
                               "stronger reranker (~2.3 GB download); gate threshold must be re-tuned"),
    "embed_bge_base":         ({"embed_model": "BAAI/bge-base-en-v1.5"}, "larger embedding model"),
}

EVAL_PLAN = {
    # retrieval mode: about 1-2 minutes per experiment on a T4, no LLM calls
    "retrieval": ["baseline", "v22_like", "no_title_pinning", "no_neighbour_expansion", "no_seed_floor",
                  "pin_rules_v25", "bm25_stemming", "continuations", "all_fixes", "dense_only",
                  "dense_only_k64", "bm25_only", "budget_2600"],
    # model swaps download extra models; run them with CONFIG["llm_load_in_4bit"] = True on a T4
    # (with the fp16 LLM loaded the reranker falls back to the CPU and takes hours)
    "retrieval_model_swaps": [],          # e.g. ["reranker_v2_m3", "embed_bge_base"]
    # full mode generates an answer per question: roughly 30-45 minutes per experiment on a T4
    "full": [],                           # e.g. ["baseline", "all_fixes"]
}

RUNS = {}
for _name in EVAL_PLAN["retrieval"] + EVAL_PLAN["retrieval_model_swaps"]:
    RUNS[(_name, "retrieval")] = run_experiment(_name, EXPERIMENTS[_name][0], mode="retrieval")
for _name in EVAL_PLAN["full"]:
    RUNS[(_name, "full")] = run_experiment(_name, EXPERIMENTS[_name][0], mode="full")

experiment_table(RUNS.values())

In [ ]:
# ================================================================
# 10.5) Report — every experiment against the baseline of the same mode
# ================================================================
REPORT_METRICS = ["ctx_fact_recall", "ctx_section_recall", "gold_rr", "answered", "refused",
                  "answer_fact_recall", "false_refusal", "refusal_ok", "faithful", "unsupported_figures",
                  "grounding", "error", "context_tokens"]
# To compare against a baseline saved in an earlier session instead of this session's run:
#   RUNS[("baseline", "full")] = load_run_file("eval_runs/baseline__full.json")   # after upload_runs()

for (_mode,) in {(m,) for (_, m) in RUNS}:
    if ("baseline", _mode) in RUNS:
        print(f"=== Baseline scorecard [{_mode}]")
        print(scorecard(RUNS[("baseline", _mode)]).to_string(index=False))

for (_name, _mode), _run in RUNS.items():
    _base = RUNS.get(("baseline", _mode))
    if _base is None or _name == "baseline":
        continue
    print(f"\n=== {_name} [{_mode}] vs baseline [{_mode}] — {EXPERIMENTS[_name][1]}")
    print(compare_runs(_base, _run, REPORT_METRICS).to_string(index=False))
    _reg = regressions(_base, _run, "answer_fact_recall" if _mode == "full" else "ctx_fact_recall")
    if len(_reg):
        print("  questions that got worse:\n" + _reg.round(2).to_string())

if ("baseline", "retrieval") in RUNS:
    print("\n=== Baseline by question type (retrieval mode)")
    print(summarize(RUNS[("baseline", "retrieval")], by="type")[["ctx_fact_recall", "ctx_section_recall", "gold_rr",
                                                                "answered", "refused"]].to_string())
    print("\n=== Baseline: heading words vs everyday wording")
    print(summarize(RUNS[("baseline", "retrieval")], by="paraphrase")[["ctx_fact_recall", "gold_rr"]].to_string())
    print("\n=== Gate threshold, recomputed from the baseline's saved scores")
    print(routing_sweep(RUNS[("baseline", "retrieval")]).to_string(index=False))

# Runs are saved in eval_runs/ (one JSON per experiment). In Colab they disappear when the runtime
# is recycled: export_runs() downloads them as a zip, upload_runs() brings them back.

## 11. Notes for going further

- **Complete exclusion list:** Section 10.1 with its sub-sections (Excl01–Excl18 and the
  24-month disease lists) is about 2,060 tokens, more than the 1,500-token budget. With
  `CONFIG["llm_load_in_4bit"] = True` the GPU has room for `context_token_budget` around 2,600,
  which fits the whole list; rerun Sections 10 and 10.1 after changing either setting.
- **If CUDA out-of-memory persists** after restarting the runtime: check the `[GPU …]` line
  printed after model loading. If an "LLM modules were placed on CPU/disk" warning appears, or
  free memory is under ~1 GB, set `CONFIG["llm_load_in_4bit"] = True` and rerun from the top.
  4-bit changes the model's arithmetic slightly, so rerun Section 10.1 afterwards to confirm the
  answers still match the original's evidence.
- **Query rewriting:** if eval shows recall misses on paraphrased user language, add one
  LangGraph retry loop that rewrites the query with the LLM when the gate fails.
- **Multi-policy scale:** move to a vector store with metadata filtering (Qdrant / pgvector)
  so each query searches only its own policy — a correctness requirement, not performance.
- **Promoting a change:** after Section 10.5, make a switch the default only if it is better with
  no clear regression, then confirm it in full mode.
- **Judge scores and tracing:** RAGAS and Langfuse integrations exist in v2.4/v2.5; add them back
  once this deterministic workflow is reliable.
- **Growing the test set:** add real user questions, each with gold sections, key facts and a
  reference answer; `check_testset()` validates them.
- **Serving split:** Sections 1–4 are ingestion; 5–9 are serving. The persisted FAISS index and
  chunks.json are the handoff artifacts when moving off Colab.


## 12. Evaluation run: collect the results for review

Section 10.5 has already run the 13 retrieval experiments. Section 12 adds the rest and packages it.

| Step | What it does | Time on a T4 |
|---|---|---|
| 12.1 | Settings (`EVAL_RUN`); records the GPU, library and model versions, a fingerprint of the policy file and all `CONFIG` values | seconds |
| 12.2 | Health checks: the test set against the loaded policy, prompt size per question, the retrieval run repeated to confirm identical results | ~2 min |
| 12.3 | Full answers to all 94 questions with the current defaults, saved every 10 questions; 3 answers repeated word for word | ~45–55 min |
| 12.4 | Summary tables, `report.md`, one zip download | seconds |

**What the zip contains:** `report.md` (start here: scorecards, results by question type, every
experiment against the baseline, weakest answers, faithfulness flags, refusals); `summary/*.csv`
(`answers_review.csv` has every answer with its missing facts and flags; `retrieval_review.csv` every
context); `runs/*.json` (raw results of every run); `environment.json`, `checks.json`, `run_log.txt`.

In [ ]:
# ================================================================
# 12.1) Evaluation run — settings, environment record, preflight
# ================================================================
EVAL_RUN = {
    "full_baseline": True,      # answer the test questions with the current system (~40-55 min on a T4)
    "full_subset": None,        # e.g. 30 = quicker first look: 30 questions spread over all types
    "full_all_fixes": False,    # also answer them with the three fixes switched on (~40-55 min more)
    "determinism_check": True,  # repeat the retrieval run and 3 answers, and compare (~3 min)
    "save_to_drive": False,     # copy results to Google Drive as they are produced (survives disconnects)
    "checkpoint_every": 10,     # full mode saves a checkpoint after this many questions
}

import platform, shutil, hashlib, datetime, importlib.metadata as _md
RESULTS_DIR = Path("rag_eval_results")
(RESULTS_DIR / "summary").mkdir(parents=True, exist_ok=True)
RUN_LOG = []                     # problems met while collecting results (saved in the zip)
_T_EVAL_START = time.perf_counter()
DRIVE_DIR = None

def _log(msg: str):
    print("  NOTE:", msg)
    RUN_LOG.append(msg)

if EVAL_RUN["save_to_drive"] and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/insurance_rag_eval")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        for p in DRIVE_DIR.glob("*.partial.json"):        # resume an interrupted full run
            shutil.copy(p, EVAL_DIR / p.name)
            print("  resuming from Drive checkpoint:", p.name)
    except Exception as e:
        _log(f"Google Drive not available ({type(e).__name__}: {e}); results stay in this runtime only")

def _mirror(path: Path):
    if DRIVE_DIR is not None:
        try:
            shutil.copy(path, DRIVE_DIR / Path(path).name)
        except Exception as e:
            _log(f"copy to Drive failed for {Path(path).name}: {e}")

def _version(pkg: str):
    try:
        return _md.version(pkg)
    except Exception:
        return None

def _revision(model):
    """Hugging Face commit hash of a loaded model, if the library recorded it."""
    configs = [getattr(model, "config", None)]
    try:                                        # SentenceTransformer: first module wraps the model
        configs.append(model[0].auto_model.config)
    except Exception:
        pass
    for cfg in configs:
        h = getattr(cfg, "_commit_hash", None)
        if h:
            return h
    return None

ENV = {
    "notebook_version": NOTEBOOK_VERSION, "testset_version": TESTSET_VERSION,
    "started": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "packages": {p: _version(p) for p in ["torch", "transformers", "sentence-transformers", "faiss-cpu",
                                          "rank-bm25", "tiktoken", "langgraph", "numpy", "pandas",
                                          "accelerate", "bitsandbytes", "snowballstemmer"]},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(torch.cuda.mem_get_info()[1] / 1e9, 1) if torch.cuda.is_available() else None,
    "models": {"embed": CONFIG["embed_model"], "embed_revision": _revision(embed_model),
               "reranker": CONFIG["reranker_model"], "reranker_revision": _revision(reranker),
               "llm": CONFIG["llm_model"], "llm_revision": _revision(model_llm),
               "llm_precision": "4-bit" if CONFIG["llm_load_in_4bit"] else "fp16"},
    "policy_file": Path(SOURCE_PATH).name,
    "policy_sha256": hashlib.sha256(Path(SOURCE_PATH).read_bytes()).hexdigest(),
    "policy_chars_after_cleaning": len(text),
    "sections": len(sections), "chunks": len(final_chunks),
    "continuation_sections_found": CONTINUATION_OF,
    "config": _config_snapshot(),
    "eval_run": dict(EVAL_RUN),
}
(RESULTS_DIR / "environment.json").write_text(json.dumps(ENV, indent=1, ensure_ascii=False))
_mirror(RESULTS_DIR / "environment.json")

print(f"GPU: {ENV['gpu']} ({ENV['gpu_memory_gb']} GB) | LLM {ENV['models']['llm_precision']} | "
      f"policy {ENV['policy_file']} (sha256 {ENV['policy_sha256'][:12]}…) | {ENV['sections']} sections, "
      f"{ENV['chunks']} chunks | test set {TESTSET_VERSION}")
if not torch.cuda.is_available() and EVAL_RUN["full_baseline"]:
    _log("no GPU: answering every question on the CPU would take many hours. Switch the runtime to a "
         "T4 GPU, or set EVAL_RUN['full_baseline'] = False to collect retrieval results only.")
_n_full = EVAL_RUN["full_subset"] or len(TESTSET)
print(f"Planned: full answers for {_n_full if EVAL_RUN['full_baseline'] else 0} questions"
      f"{' (+ the same with all fixes)' if EVAL_RUN['full_all_fixes'] else ''}; "
      f"expect about {round(_n_full * 0.5) * (2 if EVAL_RUN['full_all_fixes'] else 1)} minutes on a T4.")

In [ ]:
# ================================================================
# 12.2) Health checks — test set, prompt sizes, retrieval determinism
# ================================================================
import io, contextlib
CHECKS = {}

with contextlib.redirect_stdout(io.StringIO()):
    _problems = check_testset()
CHECKS["testset_problems"] = [list(map(str, p)) for p in _problems]
print("Test set against the loaded policy:", "OK" if not _problems else f"{len(_problems)} problem(s)")
for p in _problems:
    print("  ", p)

# Prompt size per question (LLM tokens, chat template included) for the baseline contexts
_base_ret = RUNS.get(("baseline", "retrieval"))
_prompt_rows = []
if _base_ret is None:
    _log("no baseline retrieval run found: run Section 10.5 first")
else:
    for r in _base_ret["records"]:
        try:
            n_tok = int(build_llm_inputs(build_prompt(r["question"], r["groups"]))["input_ids"].shape[1])
        except Exception as e:
            n_tok = None
            _log(f"prompt size for {r['id']}: {type(e).__name__}: {e}")
        _prompt_rows.append({"id": r["id"], "type": r["type"], "context_tokens": r["context_tokens"],
                             "prompt_tokens": n_tok})
    free_gpu()
PROMPT_SIZES = pd.DataFrame(_prompt_rows)
if len(PROMPT_SIZES):
    _pt = PROMPT_SIZES["prompt_tokens"].dropna()
    CHECKS["prompt_tokens"] = {"max": int(_pt.max()), "mean": round(float(_pt.mean()), 1),
                               "over_1700": int((_pt > 1700).sum())}
    print(f"Prompt tokens per question: mean {CHECKS['prompt_tokens']['mean']}, max {CHECKS['prompt_tokens']['max']}, "
          f"{CHECKS['prompt_tokens']['over_1700']} over 1,700 (out-of-memory risk on a T4 with fp16)")

# Determinism: the same retrieval run twice must give identical candidates, scores and contexts
if EVAL_RUN["determinism_check"] and _base_ret is not None:
    _rep = run_experiment("baseline_repeat", {}, mode="retrieval", save=False, verbose=False)
    _diff = [a["id"] for a, b in zip(_base_ret["records"], _rep["records"])
             if a["context_chunks"] != b["context_chunks"] or a["metrics"] != b["metrics"]
             or [x[0] for x in a["ranked"]] != [x[0] for x in b["ranked"]]]
    _max_score_diff = max((abs(x[1] - y[1]) for a, b in zip(_base_ret["records"], _rep["records"])
                           for x, y in zip(a["ranked"], b["ranked"]) if x[0] == y[0]), default=0.0)
    CHECKS["retrieval_repeat_identical"] = not _diff
    CHECKS["retrieval_repeat_changed_questions"] = _diff
    CHECKS["retrieval_repeat_max_score_diff"] = round(float(_max_score_diff), 6)
    print(f"Retrieval repeated: {'identical' if not _diff else f'{len(_diff)} question(s) differ: {_diff}'}"
          f" (largest reranker score difference {_max_score_diff:.6f})")
gpu_report("after health checks")

In [ ]:
# ================================================================
# 12.3) Full answers — baseline (and optionally all fixes), saved in checkpoints
# ================================================================
def _spread_subset(testset, n):
    """n questions taken in turn from each question type (same choice every time)."""
    by_type = {}
    for t in testset:
        by_type.setdefault(t["type"], []).append(t)
    picked, lists = [], [list(v) for v in by_type.values()]
    while len(picked) < n and any(lists):
        for lst in lists:
            if lst and len(picked) < n:
                picked.append(lst.pop(0))
    return sorted(picked, key=lambda t: [x["id"] for x in testset].index(t["id"]))

def run_full_checkpointed(name: str, overrides: dict, testset, every: int = 10) -> dict:
    """Full-mode run saved every `every` questions; re-running the cell resumes where it stopped."""
    part = EVAL_DIR / f"{name}__full.partial.json"
    prev = json.loads(part.read_text()) if part.exists() else None
    if prev and prev.get("testset_version") != TESTSET_VERSION:
        prev = None
    records = prev["records"] if prev else []
    done = {r["id"] for r in records}
    todo = [t for t in testset if t["id"] not in done]
    if done:
        print(f"  {name}: resuming, {len(done)} answers already saved")
    run = prev
    t0 = time.perf_counter()
    for k in range(0, len(todo), every):
        batch = run_experiment(name, overrides, mode="full", testset=todo[k:k + every], save=False, verbose=False)
        records += batch["records"]
        run = {**batch, "records": records}
        part.write_text(json.dumps(run, ensure_ascii=False))
        _mirror(part)
        left = len(todo) - (k + every)
        rate = (time.perf_counter() - t0) / min(k + every, len(todo))
        print(f"  {name} [full]: {len(records)}/{len(testset)} answered"
              + (f", about {max(left, 0) * rate / 60:.0f} min left" if left > 0 else ""))
    if run is None:
        raise RuntimeError("no questions to answer")
    order = {t["id"]: n for n, t in enumerate(testset)}
    run["records"] = sorted(records, key=lambda r: order.get(r["id"], 1e9))
    run["seconds"] = round(sum(r["seconds"] for r in run["records"]), 1)
    path = save_run(run)
    _mirror(path)
    part.unlink(missing_ok=True)
    if DRIVE_DIR is not None:
        (DRIVE_DIR / part.name).unlink(missing_ok=True)
    return run

_full_set = _spread_subset(TESTSET, EVAL_RUN["full_subset"]) if EVAL_RUN["full_subset"] else TESTSET
if EVAL_RUN["full_baseline"]:
    try:
        RUNS[("baseline", "full")] = run_full_checkpointed("baseline", {}, _full_set, EVAL_RUN["checkpoint_every"])
    except Exception as e:
        _log(f"full baseline stopped: {type(e).__name__}: {e} (re-run this cell to resume)")
if EVAL_RUN["full_all_fixes"]:
    try:
        RUNS[("all_fixes", "full")] = run_full_checkpointed("all_fixes", EXPERIMENTS["all_fixes"][0], _full_set,
                                                           EVAL_RUN["checkpoint_every"])
    except Exception as e:
        _log(f"full all_fixes run stopped: {type(e).__name__}: {e} (re-run this cell to resume)")

# Determinism of answers: answer 3 questions again and compare word for word
if EVAL_RUN["determinism_check"] and ("baseline", "full") in RUNS:
    _base_full = RUNS[("baseline", "full")]
    _ids = [r["id"] for r in _base_full["records"] if r["answerable"]][:3]
    _again = run_experiment("baseline_repeat", {}, mode="full", save=False, verbose=False,
                            testset=[t for t in TESTSET if t["id"] in _ids])
    _first = {r["id"]: r["answer"] for r in _base_full["records"]}
    _same = {r["id"]: r["answer"] == _first[r["id"]] for r in _again["records"]}
    CHECKS["answers_repeat_identical"] = _same
    print("Answers repeated word for word:", _same)

if ("baseline", "full") in RUNS:
    print(scorecard(RUNS[("baseline", "full")]).to_string(index=False))
gpu_report("after full runs")
if torch.cuda.is_available():
    CHECKS["peak_gpu_memory_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)

In [ ]:
# ================================================================
# 12.4) Package the results — summary tables, report.md, one zip to download
# ================================================================
SUMMARY = RESULTS_DIR / "summary"

def _md_table(df: pd.DataFrame, max_rows: int = None) -> str:
    if df is None or len(df) == 0:
        return "_(no data)_\n"
    df = df.head(max_rows) if max_rows else df
    fmt = lambda v: (f"{v:.3f}" if isinstance(v, float) and not float(v).is_integer() else
                     str(int(v)) if isinstance(v, float) else str(v)).replace("|", "/").replace("\n", " ")
    cols = [str(c) for c in df.columns]
    rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    rows += ["| " + " | ".join(fmt(v) if pd.notna(v) else "" for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(rows) + "\n"

def _save(df: pd.DataFrame, name: str):
    if df is not None and len(df):
        df.to_csv(SUMMARY / name, index=False)

def _missing(facts, text):
    t = norm_text(text or "")
    return [f[0] for f in facts if not fact_found(f, t)]

ITEM = {t["id"]: t for t in TESTSET}
REPORT = {}
_base = {m: RUNS.get(("baseline", m)) for m in ("retrieval", "full")}

def _step(name, fn):
    try:
        REPORT[name] = fn()
    except Exception as e:
        _log(f"{name}: {type(e).__name__}: {e}")
        REPORT[name] = None

# 1. Experiments and comparisons with the baseline of the same mode
def _experiments():
    t = experiment_table(RUNS.values())
    _save(t, "experiments.csv")
    return t
def _comparisons():
    out = []
    for (name, mode), run in RUNS.items():
        if name == "baseline" or _base.get(mode) is None:
            continue
        c = compare_runs(_base[mode], run, REPORT_METRICS)
        c.columns = ["metric", "n", "baseline", "experiment", *c.columns[4:]]
        c.insert(0, "mode", mode)
        c.insert(0, "experiment_name", name)
        out.append(c)
    df = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    _save(df, "comparisons.csv")
    return df
_step("experiments", _experiments)
_step("comparisons", _comparisons)

# 2. Baseline views
for _mode in ("retrieval", "full"):
    if _base[_mode] is not None:
        _step(f"scorecard_{_mode}", lambda m=_mode: scorecard(_base[m]))
        _step(f"by_type_{_mode}", lambda m=_mode: summarize(_base[m], by="type").reset_index())
        _step(f"by_wording_{_mode}", lambda m=_mode: summarize(_base[m], by="paraphrase").reset_index())
        for k in (f"scorecard_{_mode}", f"by_type_{_mode}", f"by_wording_{_mode}"):
            _save(REPORT[k], f"{k}.csv")
if _base["retrieval"] is not None:
    _step("routing_sweep", lambda: routing_sweep(_base["retrieval"]))
    _save(REPORT["routing_sweep"], "routing_sweep.csv")

# 3. Per-question review tables
def _retrieval_review():
    rows = []
    for r in _base["retrieval"]["records"]:
        it = ITEM[r["id"]]
        top = []
        for i, s in r["ranked"][:5]:
            top.append(f"{final_chunks[i]['section_id']} ({s:.2f})")
        rows.append({"id": r["id"], "type": r["type"], "paraphrase": r["paraphrase"],
                     "answerable": r["answerable"], "question": r["question"],
                     "gold_sections": ", ".join(it["gold"]),
                     "context_sections": ", ".join(dict.fromkeys(final_chunks[i]["section_id"] for i in r["context_chunks"])),
                     "top5_reranked": ", ".join(top), "top_score": r["top_score"], "gate": r["gate_route"],
                     "ctx_fact_recall": r["metrics"].get("ctx_fact_recall"),
                     "facts_missing_from_context": "; ".join(_missing(it["facts"], r["context"])),
                     "context_tokens": r["context_tokens"], "error": r.get("error")})
    df = pd.DataFrame(rows)
    _save(df, "retrieval_review.csv")
    return df
def _answers_review():
    rows = []
    for r in _base["full"]["records"]:
        it, m = ITEM[r["id"]], r["metrics"]
        ans = r["answer"] or ""
        rows.append({"id": r["id"], "type": r["type"], "paraphrase": r["paraphrase"],
                     "answerable": r["answerable"], "question": r["question"], "answer": ans,
                     "answer_fact_recall": m.get("answer_fact_recall"), "ctx_fact_recall": m.get("ctx_fact_recall"),
                     "facts_missing_from_answer": "; ".join(_missing(it["facts"], ans)),
                     "facts_missing_from_context": "; ".join(_missing(it["facts"], r["context"])),
                     "refused": is_no_answer(ans), "refusal_ok": m.get("refusal_ok"),
                     "unsupported_figures": "; ".join(f"{k} {v}" for k, v in sorted(unsupported_figures(ans, r["context"], r["question"]))),
                     "unsupported_citations": ", ".join(sorted(unsupported_citations(ans, r["context"]))),
                     "grounding": m.get("grounding"), "answer_words": m.get("answer_words"),
                     "context_sections": ", ".join(dict.fromkeys(final_chunks[i]["section_id"] for i in r["context_chunks"])),
                     "gold_sections": ", ".join(it["gold"]), "top_score": r["top_score"],
                     "seconds": r["seconds"], "error": r.get("error"), "oom_retry": r.get("oom_retry"),
                     "reference": it["reference"]})
    df = pd.DataFrame(rows)
    _save(df, "answers_review.csv")
    return df
if _base["retrieval"] is not None:
    _step("retrieval_review", _retrieval_review)
if _base["full"] is not None:
    _step("answers_review", _answers_review)

# 4. Older checks kept from earlier versions, and the health checks
for _name, _var in (("gold_eval", "gold_df"), ("regression_check", "regression_df")):
    if _var in globals():
        REPORT[_name] = globals()[_var]
        _save(REPORT[_name], f"{_name}.csv")
_save(PROMPT_SIZES, "prompt_sizes.csv")
CHECKS["minutes_for_section_12"] = round((time.perf_counter() - _T_EVAL_START) / 60, 1)
(RESULTS_DIR / "checks.json").write_text(json.dumps(CHECKS, indent=1, default=str))

# 5. report.md: the same results in one readable file
def _report_md() -> str:
    E, L = ENV, []
    L.append(f"# RAG evaluation results — {E['notebook_version']}\n")
    L.append(f"- Run: {E['started']} | GPU: {E['gpu']} | LLM: {E['models']['llm_precision']}"
             f" | test set {E['testset_version']} | policy sha256 {E['policy_sha256'][:12]}… "
             f"| {E['sections']} sections, {E['chunks']} chunks\n")
    L.append("## 1. Checks\n")
    L.append(f"- Test set against the policy: {'OK' if not CHECKS.get('testset_problems') else CHECKS['testset_problems']}")
    L.append(f"- Retrieval repeated identically: {CHECKS.get('retrieval_repeat_identical')} "
             f"(largest score difference {CHECKS.get('retrieval_repeat_max_score_diff')})")
    L.append(f"- Answers repeated word for word: {CHECKS.get('answers_repeat_identical')}")
    L.append(f"- Prompt tokens: {CHECKS.get('prompt_tokens')} | peak GPU memory (GB): {CHECKS.get('peak_gpu_memory_gb')}")
    if _base["full"] is not None:
        rt = run_table(_base["full"])
        L.append(f"- Full baseline: {len(rt)} questions, pipeline errors {int(rt['error'].sum())}, "
                 f"out-of-memory retries {int(rt.get('oom_retry', pd.Series(dtype=float)).sum())}, "
                 f"{round(_base['full']['seconds'] / 60, 1)} min of answering")
    L.append("")
    for mode in ("retrieval", "full"):
        if REPORT.get(f"scorecard_{mode}") is not None:
            L.append(f"## 2{'a' if mode == 'retrieval' else 'b'}. Baseline scorecard ({mode} mode)\n")
            L.append(_md_table(REPORT[f"scorecard_{mode}"]))
    for mode in ("retrieval", "full"):
        bt = REPORT.get(f"by_type_{mode}")
        if bt is not None:
            keep = [c for c in ["type", "ctx_fact_recall", "gold_rr", "answered", "refused", "answer_fact_recall",
                                "false_refusal", "refusal_ok", "faithful", "unsupported_figures"] if c in bt]
            L.append(f"## 3{'a' if mode == 'retrieval' else 'b'}. Baseline by question type ({mode} mode)\n")
            L.append(_md_table(bt[keep]))
            bw = REPORT.get(f"by_wording_{mode}")
            if bw is not None:
                L.append("Heading words (`paraphrase=False`) vs everyday wording (`True`):\n")
                L.append(_md_table(bw[[c for c in ["paraphrase", "ctx_fact_recall", "gold_rr", "answer_fact_recall"] if c in bw]]))
    comp = REPORT.get("comparisons")
    if comp is not None and len(comp):
        L.append("## 4. Experiments vs the baseline\n")
        main = comp[comp["metric"].isin(["ctx_fact_recall", "answer_fact_recall", "answered", "refused",
                                          "faithful", "context_tokens"])]
        L.append(_md_table(main[["experiment_name", "mode", "metric", "baseline", "experiment", "diff",
                                 "ci_low", "ci_high", "questions_up", "questions_down", "verdict"]]))
    if REPORT.get("routing_sweep") is not None:
        L.append("## 5. Gate threshold (recomputed from the baseline's scores)\n")
        L.append(_md_table(REPORT["routing_sweep"]))
    ar = REPORT.get("answers_review")
    if ar is not None:
        weak = ar[ar["answerable"]].sort_values(["answer_fact_recall", "ctx_fact_recall"]).head(20)
        L.append("## 6. Weakest answers (full baseline)\n")
        L.append(_md_table(weak[["id", "type", "answer_fact_recall", "ctx_fact_recall", "facts_missing_from_answer",
                                 "facts_missing_from_context", "context_sections", "gold_sections"]]))
        flags = ar[(ar["unsupported_figures"] != "") | (ar["unsupported_citations"] != "")]
        L.append("## 7. Faithfulness flags (figures or section numbers not in the context)\n")
        L.append(_md_table(flags[["id", "question", "unsupported_figures", "unsupported_citations"]]))
        neg = ar[~ar["answerable"]]
        L.append("## 8. Questions the policy cannot answer\n")
        L.append(_md_table(neg[["id", "type", "refused", "refusal_ok", "answer"]].assign(
            answer=neg["answer"].str.slice(0, 160))))
    rr = REPORT.get("retrieval_review")
    if rr is not None:
        L.append("## 9. Weakest contexts (retrieval baseline)\n")
        weakc = rr[rr["answerable"]].sort_values("ctx_fact_recall").head(15)
        L.append(_md_table(weakc[["id", "type", "ctx_fact_recall", "facts_missing_from_context",
                                  "context_sections", "gold_sections", "top_score"]]))
    for name, title in (("gold_eval", "Original gold set (Section 10)"), ("regression_check", "Regression check vs the original notebook (Section 10.1)")):
        if REPORT.get(name) is not None:
            L.append(f"## {title}\n")
            L.append(_md_table(REPORT[name]))
    L.append("## Problems during the run\n")
    L.append("\n".join(f"- {m}" for m in RUN_LOG) if RUN_LOG else "None.")
    return "\n".join(L) + "\n"

(RESULTS_DIR / "report.md").write_text(_report_md())
(RESULTS_DIR / "run_log.txt").write_text("\n".join(RUN_LOG) or "no problems")

# 6. One zip: the summary, the report, and every saved run
_stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_zip_root = Path(f"rag_eval_results_{_stamp}")
if _zip_root.exists():
    shutil.rmtree(_zip_root)
shutil.copytree(RESULTS_DIR, _zip_root)
(_zip_root / "runs").mkdir(exist_ok=True)
for p in EVAL_DIR.glob("*.json"):
    if not p.name.endswith(".partial.json"):
        shutil.copy(p, _zip_root / "runs" / p.name)
ZIP_PATH = shutil.make_archive(str(_zip_root), "zip", root_dir=".", base_dir=_zip_root.name)
_mirror(Path(ZIP_PATH))
print(f"\nResults: {ZIP_PATH} ({Path(ZIP_PATH).stat().st_size / 1e6:.1f} MB) — "
      f"{len(list((_zip_root / 'runs').glob('*.json')))} runs, report.md, summary tables")
if RUN_LOG:
    print("Problems during the run (also in run_log.txt):\n  - " + "\n  - ".join(RUN_LOG))
if IN_COLAB:
    from google.colab import files
    files.download(ZIP_PATH)